# Dataset Preprocessing

### Overview
This cell loads the LIAR dataset (test split) and prepares it for preprocessing by dropping unnecessary columns like individual vote counts, job titles, and party affiliations.

In [1]:
import pandas as pd

columns = [
    "id", "label", "statement", "subjects", "speaker", "job_title", "state_info", "party_affiliation",
    "barely_true_counts", "false_counts", "half_true_counts", "mostly_true_counts", "pants_on_fire_counts", "context"
]

df = pd.read_csv("Datasets/LIAR/test.tsv", sep="\t", names=columns, header=None)
df.drop(columns=["id", "barely_true_counts", "false_counts", "half_true_counts", "mostly_true_counts", "pants_on_fire_counts", "job_title", "state_info", "party_affiliation", "context", "speaker"], inplace=True)
df.to_csv("1_Basics_Masterthesis/LIAR/test_cleaned.csv", index=False)
df

,label,statement,subjects
0,true,Building a wall on the U.S.-Mexico border will...,immigration
1,false,Wisconsin is on pace to double the number of l...,jobs
2,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record"
3,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti..."
4,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti..."
...,...,...,...
1262,half-true,Says his budget provides the highest state fun...,education
1263,barely-true,Ive been here almost every day.,"civil-rights,crime,criminal-justice"
1264,barely-true,"In the early 1980s, Sen. Edward Kennedy secret...","bipartisanship,congress,foreign-policy,history"
1265,barely-true,Says an EPA permit languished under Strickland...,"environment,government-efficiency"


### Text Cleaning & Preprocessing
This section defines functions to clean textual data by removing emojis, URLs, hashtags, user mentions, and special characters. The `clean_tweets()` function is applied to normalize statements for consistent model input.

In [2]:
import re

# Muster für die Textbereinigung
sent_patt = re.compile(r'(?<!\.|\!|\?|\:|\;|\s)\w[.,:;!?]\s+')
multi_sym = re.compile(r'[!.,?=-]{2,}')
time = re.compile(r'([0-2][0-9]:[0-5][0-9]([pm]|[am])*)|([0-2]*[0-9]*:*[0-5][0-9]([pm]|[am])+)|([0-9][0-9]*([pm]|[am])+)')
date = re.compile(r'([0-3]*[0-9]\/[0-9]*\/[0-9]+)')


def remove_emojis(text):
    """
    Entfernt Emojis aus einem gegebenen Text.

    Args:
        text (str): Der zu verarbeitende Text.

    Returns:
        str: Der Text ohne Emojis.
    """
    emoji_pattern = re.compile(
        r"[" 
        u"\U0001F600-\U0001F64F"  # Emoticons
        u"\U0001F300-\U0001F5FF"  # Symbole & Piktogramme
        u"\U0001F680-\U0001F6FF"  # Transport & Karten
        u"\U0001F1E0-\U0001F1FF"  # Flaggen (iOS)
        u"\U00002702-\U000027B0"  # Weitere Symbole
        u"\U000024C2-\U0001F251"  # Diverse Zeichen
        u"\U0000200D"             # Zero Width Joiner
        u"\U0001F926-\U0001F937"  # Gesten
        u"\U00010000-\U0010FFFF"  # Andere Zeichen (Plane 1+)
        u"\u200d"                 # Zero Width Joiner
        u"\u2640-\u2642"          # Geschlechterzeichen
        u"\u2600-\u2B55"          # Diverse Symbole
        u"\u23cf"                 # Eject-Button
        u"\u23e9"                 # Weitere Buttons
        u"\u231a"                 # Uhr
        u"\u3030"                 # Wellenlinie
        u"\ufe0f"                 # Variation Selector
        u"\u2069"                 # Pop Directional Isolate
        u"\u2066"                 # Isolates
        u"\u200c"                 # Zero Width Non-Joiner
        u"\u25aa"                 # Kleine schwarze Quadrate
        u"\u25ab"                 # Kleine weiße Quadrate
        "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.sub(r'', text)

def clean_tweets(df):
    """
    Bereinigt Tweets in einem DataFrame, indem unerwünschte Zeichen, Links, Hashtags, Benutzerkonten und Emojis entfernt werden.

    Args:
        df (DataFrame): Ein DataFrame, der die Tweets enthält.

    Returns:
        DataFrame: Der bereinigte DataFrame.
    """
    for idx in range(len(df)):
        tweet = df.iloc[idx]['statement']
        
        tweet = " " + tweet + " "
        
        tweet = tweet.replace("\n", " ").replace("\"", "").replace(",", "").replace("!", "").replace("?", "").replace("“", "").replace("„", "").replace("”", "").replace("|", " ").replace("`", " ").replace("'", " ").replace(":_", " ").replace("_", " ").replace(" rt ", " retweet ").replace(" mrs. ", " mrs ").replace(" ms. ", " ms ").replace(" mr. ", " mr ").replace(" dr. ", " dr ").replace(" prof. ", " prof ").replace(" dr.-ing. ", " dr.-ing ")  
        tweet = re.sub(r'http\S+', ' hrefl ', tweet)  # links
        tweet = re.sub(r'#\S+', '', tweet)  # hashtag
        tweet = re.sub(r'@\S+', ' usacc ', tweet)  # useraccount
        tweet = remove_emojis(tweet)  # Emojis entfernen
        
        all_sym = multi_sym.finditer(tweet)
        all_time = time.finditer(tweet)
        all_date = date.finditer(tweet)
        
        for m in all_sym:
            tweet = tweet.replace(m.group(), ' ' + m.group()[0] + ' ', 1)
        for m in all_time:
            tweet = tweet.replace(m.group(), ' tiform ', 1)
        for m in all_date:
            tweet = tweet.replace(m.group(), ' dtform ', 1)
        tweet = " " + tweet + " "
        all_pts = sent_patt.finditer(tweet)
        for m in all_pts:
            tweet = tweet.replace(m.group(), m.group()[0] + ' ' + m.group()[1] + ' ', 1)
        
        tweet = re.sub(r"\s\s+", " ", tweet)
        df.iloc[idx, df.columns.get_loc('statement')] = tweet.strip()
    return df

In [3]:
import pandas as pd
df = pd.read_csv('1_Basics_Masterthesis/LIAR/test_cleaned.csv')

df = clean_tweets(df)
df.to_csv('1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned.csv', sep='\t', index=False)

In [4]:
df

,label,statement,subjects
0,true,Building a wall on the U.S . Mexico border wil...,immigration
1,false,Wisconsin is on pace to double the number of l...,jobs
2,false,Says John McCain has done nothing to help the ...,"military,veterans,voting-record"
3,half-true,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti..."
4,pants-fire,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti..."
...,...,...,...
1262,half-true,Says his budget provides the highest state fun...,education
1263,barely-true,Ive been here almost every day .,"civil-rights,crime,criminal-justice"
1264,barely-true,In the early 1980s Sen . Edward Kennedy secret...,"bipartisanship,congress,foreign-policy,history"
1265,barely-true,Says an EPA permit languished under Strickland...,"environment,government-efficiency"


## binary

### Binary Label Conversion
Converts multi-class labels (true, mostly-true, half-true, barely-true, false, pants-fire) into binary labels (True/False), grouping ambiguous categories for cleaner binary classification.

In [ ]:
import pandas as pd

df = pd.read_csv("1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned.csv", sep="\t")

# define Label-Mapping
label_map = {
    "true": "True",
    "mostly-true": "True",
    "half-true": "True",
    "barely-true": "False",
    "false": "False",
    "pants-fire": "False"
}

# convert Labels
df["label"] = df["label"].str.strip().str.lower().map(label_map)

# delete rows with unknown labels
df = df[df["label"].notna()]

print(df["label"].value_counts())

df.to_csv("1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv", sep="\t", index=False)

label
True     714
False    553
Name: count, dtype: int64


In [6]:
df

,label,statement,subjects
0,True,Building a wall on the U.S . Mexico border wil...,immigration
1,False,Wisconsin is on pace to double the number of l...,jobs
2,False,Says John McCain has done nothing to help the ...,"military,veterans,voting-record"
3,True,Suzanne Bonamici supports a plan that will cut...,"medicare,message-machine-2012,campaign-adverti..."
4,False,When asked by a reporter whether hes at the ce...,"campaign-finance,legal-issues,campaign-adverti..."
...,...,...,...
1262,True,Says his budget provides the highest state fun...,education
1263,False,Ive been here almost every day .,"civil-rights,crime,criminal-justice"
1264,False,In the early 1980s Sen . Edward Kennedy secret...,"bipartisanship,congress,foreign-policy,history"
1265,False,Says an EPA permit languished under Strickland...,"environment,government-efficiency"


# Testing Models without Subject

### Zero-Shot Model Evaluation (Without Subject Context)
**Setup A (Strict)**: Models are asked to respond with only "True" or "False". Outputs that don't match are classified as "UNK". This measures genuine zero-shot capability without post-processing.

**Setup B (Forced)**: All models are forced to produce binary decisions via heuristic post-processing. Unknown outputs are resolved using a fallback strategy to create complete predictions.

## Setup A

Model only accepts True or False, everything else gets marked as unknown.
Measures genuine Zero-Shot-Behaviour of the model.
-> can it decide binary question under minimal instructions?

In [ ]:
import re
import torch
import pandas as pd
from tqdm.auto import tqdm


import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForMaskedLM,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
OUT_PATH = "1_Basics_Masterthesis/Results/statements_with_predictions_all_models.csv"


MODELS = {
    # =========================
    # Decoder-only / Instruct
    # =========================
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
    "deepseek-ai/deepseek-llm-7b-base": "causal",
    "EleutherAI/gpt-j-6B": "causal",
    "google/gemma-7b-it": "causal_chat_or_instruct",
    "google/gemma-2b-it": "causal_chat_or_instruct",

    # =========================
    # Encoder-Decoder
    # =========================
    "google/flan-t5-xl": "seq2seq",
    "google/flan-t5-xxl": "seq2seq", 
    "t5-base": "seq2seq", 
    "google/flan-t5-base": "seq2seq",

    # =========================
    # Encoder-only
    # =========================
    "bert-base-uncased": "mlm",
    "bert-large-uncased": "mlm",
    "roberta-base": "mlm",
    "roberta-large": "mlm",
}


# -----------------------------
# Prompt
# -----------------------------
def build_prompt(text: str) -> str:
    return (
        f"Statement: {text}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )

# -----------------------------
# Robust: True/False from Text
# -----------------------------
def normalize_tf(text: str):
    if text is None:
        return "UNK", None
    s = text.strip().lower()
    m = re.search(r"\b(true|false)\b", s)
    if not m:
        return "UNK", None
    return ("True", 1) if m.group(1) == "true" else ("False", 0)

# -----------------------------
# use Chat-Template if existing
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            return prompt
    return prompt

# -----------------------------
# MLM (BERT/RoBERTa): Mask-Scoring
# -----------------------------
def mlm_true_false(model, tokenizer, statement: str):
    if tokenizer.mask_token is None:
        return "UNK", None

    prompt = (
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: {tokenizer.mask_token}.\n"
        f"Answer: {tokenizer.mask_token}"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits  # [B, T, V]

    mask_id = tokenizer.mask_token_id
    mask_positions = (inputs["input_ids"][0] == mask_id).nonzero(as_tuple=False).flatten()
    if mask_positions.numel() == 0:
        return "UNK", None
    pos = mask_positions[-1].item() 

    probs = torch.softmax(logits[0, pos], dim=-1)

    true_ids = tokenizer.encode(" true", add_special_tokens=False)
    false_ids = tokenizer.encode(" false", add_special_tokens=False)
    if len(true_ids) != 1:
        true_ids = tokenizer.encode("true", add_special_tokens=False)
    if len(false_ids) != 1:
        false_ids = tokenizer.encode("false", add_special_tokens=False)

    if len(true_ids) != 1 or len(false_ids) != 1:
        return "UNK", None

    p_true = probs[true_ids[0]].item()
    p_false = probs[false_ids[0]].item()

    return ("True", 1) if p_true >= p_false else ("False", 0)

# -----------------------------
# MAIN
# -----------------------------
def main():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    if "statement" not in df.columns:
        raise ValueError(f"Spalte 'statement' nicht gefunden. Spalten: {list(df.columns)}")

    statements = df["statement"].astype(str).tolist()

    for model_name in MODELS:
        short = model_name.split("/")[-1]
        df[f"pred_{short}"] = None
        df[f"pred_bin_{short}"] = None

    for model_name, mtype in MODELS.items():
        print(f"\n=== Loading: {model_name} ({mtype}) ===")
        short = model_name.split("/")[-1]

        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        if mtype in ("causal", "causal_chat_or_instruct"):
            if tokenizer.pad_token is None and tokenizer.eos_token is not None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, text in enumerate(
                    tqdm(statements, desc=f"{short} | generating", leave=True)
                ):
                prompt = build_prompt(text)
                rendered = maybe_apply_chat_template(tokenizer, prompt, mtype)
            
                inputs = tokenizer(
                    rendered,
                    return_tensors="pt",
                    truncation=True
                ).to(device)
            
                output = model.generate(
                    **inputs,
                    max_new_tokens=6,
                    do_sample=False,
                    num_beams=1,
                    temperature=None,
                    top_p=None,
                    pad_token_id=tokenizer.eos_token_id
                )

            
                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                label_str, label_bin = normalize_tf(decoded)
            
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin


        elif mtype == "seq2seq":
                    model = AutoModelForSeq2SeqLM.from_pretrained(
                        model_name,
                        dtype=torch.float16 if device.type == "cuda" else None,
                    ).to(device)
                    model.eval()
        
                    for i, text in enumerate(
                        tqdm(statements, desc=f"{short} | generating", leave=True)
                    ):
                        prompt = build_prompt(text)
        
                        inputs = tokenizer(
                            prompt,
                            return_tensors="pt",
                            truncation=True
                        ).to(device)
        
                        output = model.generate(
                            **inputs,
                            max_new_tokens=6,
                            do_sample=False,
                            num_beams=1,
                            temperature=None,
                            top_p=None,
                            pad_token_id=tokenizer.eos_token_id
                        )

        
                        decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                        label_str, label_bin = normalize_tf(decoded)
        
                        df.at[df.index[i], f"pred_{short}"] = label_str
                        df.at[df.index[i], f"pred_bin_{short}"] = label_bin



        elif mtype == "mlm":
            model = AutoModelForMaskedLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, text in enumerate(
                    tqdm(
                        statements,
                        desc=f"{short} | scoring",
                        leave=True
                    )
                ):
                label_str, label_bin = mlm_true_false(model, tokenizer, text)
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin
        else:
            raise ValueError(f"Unbekannter Modelltyp: {mtype}")

        # cleanup
        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    df.to_csv(OUT_PATH, index=False)
    print(f"\nFertig. Gespeichert: {OUT_PATH}")

if __name__ == "__main__":
    main()




=== Lade: meta-llama/Llama-3.1-8B-Instruct (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | generating:   0%|          | 0/1267 [00:00<?, ?it/s]


=== Lade: deepseek-ai/deepseek-llm-7b-base (causal) ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

roberta-large | scoring:   0%|          | 0/1267 [00:00<?, ?it/s]


Fertig. Gespeichert: 1_Basics_Masterthesis/Results/statements_with_predictions_all_models.csv


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/Results/statements_with_predictions_all_models.csv"
LABEL_COL = "label"
OUT_SUMMARY = "1_Basics_Masterthesis/Results/model_metrics_summary.csv"

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)

if LABEL_COL not in df.columns:
    raise ValueError(f"Label-Spalte '{LABEL_COL}' nicht gefunden!")

y_true = df[LABEL_COL].astype(int)

pred_cols = [c for c in df.columns if c.startswith("pred_bin_")]

print(f"Gefundene Modelle: {len(pred_cols)}")
print(pred_cols)

# -----------------------------
# EVALUATION
# -----------------------------
rows = []

for col in pred_cols:
    model_name = col.replace("pred_bin_", "")

    y_pred = df[col]

    mask = y_pred.notna()
    y_true_f = y_true[mask]
    y_pred_f = y_pred[mask].astype(int)

    acc = accuracy_score(y_true_f, y_pred_f)
    cm = confusion_matrix(y_true_f, y_pred_f, labels=[0, 1])

    print("\n" + "=" * 60)
    print(f"Model: {model_name}")
    print(f"Samples used: {len(y_true_f)}")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true_f, y_pred_f, digits=4))

    rows.append({
        "model": model_name,
        "accuracy": acc,
        "n_samples": len(y_true_f),
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    })

# -----------------------------
# SUMMARY TABLE
# -----------------------------
summary_df = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
summary_df.to_csv(OUT_SUMMARY, index=False)

print("\n==============================")
print("Summary saved to:")
print(OUT_SUMMARY)
print(summary_df)


Gefundene Modelle: 13
['pred_bin_Llama-3.1-8B-Instruct', 'pred_bin_deepseek-llm-7b-base', 'pred_bin_gpt-j-6B', 'pred_bin_gemma-7b-it', 'pred_bin_gemma-2b-it', 'pred_bin_flan-t5-xl', 'pred_bin_flan-t5-xxl', 'pred_bin_t5-base', 'pred_bin_flan-t5-base', 'pred_bin_bert-base-uncased', 'pred_bin_bert-large-uncased', 'pred_bin_roberta-base', 'pred_bin_roberta-large']

Model: Llama-3.1-8B-Instruct
Samples used: 1267
Accuracy: 0.5635
Confusion Matrix (rows=true, cols=pred):
[[  1 552]
 [  1 713]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5000    0.0018    0.0036       553
           1     0.5636    0.9986    0.7206       714

    accuracy                         0.5635      1267
   macro avg     0.5318    0.5002    0.3621      1267
weighted avg     0.5359    0.5635    0.4076      1267


Model: deepseek-llm-7b-base
Samples used: 1267
Accuracy: 0.5635
Confusion Matrix (rows=true, cols=pred):
[[  1 552]
 [  1 713]]

Classification Report:
   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Results indicate that most llms perfom poorly at zero-shot binary fact-checking on the LIAR dataset, when evaluated under a strict evaluation protocol.
Although several models achieve accuracies around 56–58%, this performance is misleading. A closer inspection of the confusion matrices reveals that many models—especially large decoder-only instruction-tuned models—almost always predict the majority class (“True”), rather than genuinely distinguishing between true and false statements.

1. Majority-Class Collapse in Large Decoder Models

        Models such as:
        
        LLaMA-3.1-8B-Instruct
        DeepSeek-7B-Base
        GPT-J-6B
        Gemma-2B-IT and Gemma-7B-IT
        
        exhibit near-identical confusion matrices, predicting “True” in almost all cases.
        Their accuracy (~56.3%) closely matches the proportion of true statements in the dataset, indicating a trivial majority-class strategy rather than meaningful fact verification.
        
        These models show extremely low recall for the “False” class, making their predictions unreliable despite seemingly acceptable accuracy values.


2. Accuracy Is Not a Reliable Metric Here

        Because the dataset is slightly imbalanced, a model that always predicts “True” already achieves around 56% accuracy.
        Therefore, accuracy alone overestimates performance and masks the models’ inability to detect false claims.
        
        More informative metrics (e.g., balanced accuracy or macro-F1) reveal that these models perform close to chance level.


3. Encoder-Decoder Models Show More Balanced Behavior

        Models such as FLAN-T5-XL and FLAN-T5-XXL demonstrate a more balanced prediction pattern, producing both true and false outputs.
        
        FLAN-T5-XL achieves the highest accuracy (≈57.9%) without collapsing entirely to the majority class.
        
        However, even these models still struggle, indicating that zero-shot fact-checking remains a difficult task.


4. Encoder-Only Models Are Stable but Weak

        BERT and RoBERTa models show:
        
        More balanced confusion matrices
        
        Lower overall accuracy
        
        This suggests that encoder-only architectures are less biased, but also lack sufficient factual reasoning capability in a zero-shot setting.

The evaluation shows that, under strict zero-shot conditions, most large language models fail to reliably distinguish between true and false statements, frequently collapsing to majority-class predictions despite achieving moderate accuracy scores.

## Setup B

Model gets forced to decide between True/False, unclear outputs are resolved heuristically.
No clean Zero-Shot-Behaviour, but model + heuristic.
-> How does model perform if it gets forced zu decide?

In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForMaskedLM,
)

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
OUT_PATH = "1_Basics_Masterthesis/Results/statements_with_predictions_setupB.csv"

LABEL_COL = "label"

# SETUP B Heuristic:
FALLBACK_LABEL_STR = "True"   # or "False"
FALLBACK_LABEL_BIN = 1 if FALLBACK_LABEL_STR == "True" else 0
TAKE_OCCURRENCE = "last"

MODELS = {
    # =========================
    # Decoder-only / Instruct
    # =========================
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
    "deepseek-ai/deepseek-llm-7b-base": "causal",
    "EleutherAI/gpt-j-6B": "causal",
    "google/gemma-7b-it": "causal_chat_or_instruct",
    "google/gemma-2b-it": "causal_chat_or_instruct",

    # =========================
    # Encoder-Decoder
    # =========================
    "google/flan-t5-xl": "seq2seq",
    "google/flan-t5-xxl": "seq2seq",
    "t5-base": "seq2seq", 
    "google/flan-t5-base": "seq2seq",

    # =========================
    # Encoder-only
    # =========================
    "bert-base-uncased": "mlm",
    "bert-large-uncased": "mlm",
    "roberta-base": "mlm",
    "roberta-large": "mlm",
}


# -----------------------------
# Prompt
# -----------------------------
def build_prompt(text: str) -> str:
    return (
        f"Statement: {text}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )

# -----------------------------
# Setup B: "forced" Parser
# - detects true/false robustly
# - if not detectable => fallback (default True)
# -----------------------------
def parse_true_false_forced(text: str):
    """
    Setup B: erzwingt eine binäre Entscheidung.
    Gibt (label_str, label_bin) zurück.
    """
    if text is None:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN

    s = str(text).strip().lower()
    
    matches = re.findall(r"\b(true|false)\b", s)

    if not matches:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN

    choice = matches[0] if TAKE_OCCURRENCE == "first" else matches[-1]
    if choice == "true":
        return "True", 1
    else:
        return "False", 0

# -----------------------------
# use Chat-Template if existing
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            return prompt
    return prompt

# -----------------------------
# MLM: Mask-Scoring same as Setup A
# -----------------------------
def mlm_true_false(model, tokenizer, statement: str):
    if tokenizer.mask_token is None:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN

    prompt = (
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: {tokenizer.mask_token}.\n"
        f"Answer: {tokenizer.mask_token}"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits

    mask_id = tokenizer.mask_token_id
    mask_positions = (inputs["input_ids"][0] == mask_id).nonzero(as_tuple=False).flatten()
    if mask_positions.numel() == 0:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN
    pos = mask_positions[-1].item()

    probs = torch.softmax(logits[0, pos], dim=-1)

    true_ids = tokenizer.encode(" true", add_special_tokens=False)
    false_ids = tokenizer.encode(" false", add_special_tokens=False)
    if len(true_ids) != 1:
        true_ids = tokenizer.encode("true", add_special_tokens=False)
    if len(false_ids) != 1:
        false_ids = tokenizer.encode("false", add_special_tokens=False)

    if len(true_ids) != 1 or len(false_ids) != 1:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN

    p_true = probs[true_ids[0]].item()
    p_false = probs[false_ids[0]].item()
    return ("True", 1) if p_true >= p_false else ("False", 0)

# -----------------------------
# MAIN
# -----------------------------
def run_predictions():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    if "statement" not in df.columns:
        raise ValueError(f"Spalte 'statement' nicht gefunden. Spalten: {list(df.columns)}")
    if LABEL_COL not in df.columns:
        raise ValueError(f"Label-Spalte '{LABEL_COL}' nicht gefunden. Spalten: {list(df.columns)}")

    statements = df["statement"].astype(str).tolist()

    for model_name in MODELS:
        short = model_name.split("/")[-1]
        df[f"pred_{short}"] = None
        df[f"pred_bin_{short}"] = None
        df[f"raw_{short}"] = None

    for model_name, mtype in MODELS.items():
        print(f"\n=== Lade: {model_name} ({mtype}) ===")
        short = model_name.split("/")[-1]

        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        if mtype in ("causal", "causal_chat_or_instruct"):
            if tokenizer.pad_token is None and tokenizer.eos_token is not None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, text in enumerate(tqdm(statements, desc=f"{short} | generating", leave=True)):
                prompt = build_prompt(text)
                rendered = maybe_apply_chat_template(tokenizer, prompt, mtype)

                inputs = tokenizer(rendered, return_tensors="pt", truncation=True).to(device)

                output = model.generate(
                    **inputs,
                    max_new_tokens=12,
                    do_sample=False,
                    num_beams=1,
                    pad_token_id=tokenizer.eos_token_id
                )

                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                
                label_str, label_bin = parse_true_false_forced(decoded)

                df.at[df.index[i], f"raw_{short}"] = decoded
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        elif mtype == "seq2seq":
            model = AutoModelForSeq2SeqLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, text in enumerate(tqdm(statements, desc=f"{short} | generating", leave=True)):
                prompt = build_prompt(text)
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

                output = model.generate(
                    **inputs,
                    max_new_tokens=12,
                    do_sample=False,
                    num_beams=1,
                    pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None
                )

                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                label_str, label_bin = parse_true_false_forced(decoded)

                df.at[df.index[i], f"raw_{short}"] = decoded
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        elif mtype == "mlm":
            model = AutoModelForMaskedLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, text in enumerate(tqdm(statements, desc=f"{short} | scoring", leave=True)):
                label_str, label_bin = mlm_true_false(model, tokenizer, text)

                df.at[df.index[i], f"raw_{short}"] = None
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        else:
            raise ValueError(f"Unbekannter Modelltyp: {mtype}")

        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    df.to_csv(OUT_PATH, index=False)
    print(f"\nPredictions gespeichert: {OUT_PATH}")
    return OUT_PATH

# -----------------------------
# RUN
# -----------------------------
if __name__ == "__main__":
    pred_path = run_predictions()




=== Lade: meta-llama/Llama-3.1-8B-Instruct (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | generating:   0%|          | 0/1267 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



=== Lade: deepseek-ai/deepseek-llm-7b-base (causal) ===


...


=== Lade: roberta-base (mlm) ===


roberta-base | scoring:   0%|          | 0/1267 [00:00<?, ?it/s]


=== Lade: roberta-large (mlm) ===


roberta-large | scoring:   0%|          | 0/1267 [00:00<?, ?it/s]


Predictions gespeichert: 1_Basics_Masterthesis/Results/statements_with_predictions_setupB.csv


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/Results/statements_with_predictions_setupB.csv"
LABEL_COL = "label"
OUT_SUMMARY = "1_Basics_Masterthesis/Results/model_metrics_summary_setupB.csv"

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)

if LABEL_COL not in df.columns:
    raise ValueError(f"Label-Spalte '{LABEL_COL}' nicht gefunden!")

y_true = df[LABEL_COL].astype(int)

pred_cols = [c for c in df.columns if c.startswith("pred_bin_")]

print(f"Gefundene Modelle: {len(pred_cols)}")
print(pred_cols)

# -----------------------------
# EVALUATION
# -----------------------------
rows = []

for col in pred_cols:
    model_name = col.replace("pred_bin_", "")

    y_pred = df[col]

    mask = y_pred.notna()
    y_true_f = y_true[mask]
    y_pred_f = y_pred[mask].astype(int)

    acc = accuracy_score(y_true_f, y_pred_f)
    cm = confusion_matrix(y_true_f, y_pred_f, labels=[0, 1])

    print("\n" + "=" * 60)
    print(f"Model: {model_name}")
    print(f"Samples used: {len(y_true_f)}")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true_f, y_pred_f, digits=4))

    rows.append({
        "model": model_name,
        "accuracy": acc,
        "n_samples": len(y_true_f),
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    })

# -----------------------------
# SUMMARY TABLE
# -----------------------------
summary_df = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
summary_df.to_csv(OUT_SUMMARY, index=False)

print("\n==============================")
print("Summary saved to:")
print(OUT_SUMMARY)
print(summary_df)


Gefundene Modelle: 13
['pred_bin_Llama-3.1-8B-Instruct', 'pred_bin_deepseek-llm-7b-base', 'pred_bin_gpt-j-6B', 'pred_bin_gemma-7b-it', 'pred_bin_gemma-2b-it', 'pred_bin_flan-t5-xl', 'pred_bin_flan-t5-xxl', 'pred_bin_t5-base', 'pred_bin_flan-t5-base', 'pred_bin_bert-base-uncased', 'pred_bin_bert-large-uncased', 'pred_bin_roberta-base', 'pred_bin_roberta-large']

Model: Llama-3.1-8B-Instruct
Samples used: 1267
Accuracy: 0.5288
Confusion Matrix (rows=true, cols=pred):
[[445 108]
 [489 225]]

Classification Report:
              precision    recall  f1-score   support

           0     0.4764    0.8047    0.5985       553
           1     0.6757    0.3151    0.4298       714

    accuracy                         0.5288      1267
   macro avg     0.5761    0.5599    0.5142      1267
weighted avg     0.5887    0.5288    0.5034      1267


Model: deepseek-llm-7b-base
Samples used: 1267
Accuracy: 0.4365
Confusion Matrix (rows=true, cols=pred):
[[552   1]
 [713   1]]

Classification Report:
   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Under Setup B, where every model is forced to produce a binary decision through heuristic post-processing, the results show substantially different behavior compared to the strict zero-shot evaluation (Setup A).

Overall, accuracies range from approximately 43% to 61%, with several models now producing more balanced confusion matrices, indicating that they are no longer collapsing entirely to the majority class.

Key Observations
1. Forced Decisions Mitigate Majority-Class Collapse

        In contrast to Setup A, large decoder-only models no longer predict “True” almost exclusively.
        
        For example:
        
        Gemma-7B-IT achieves the highest accuracy (≈61.2%) with a comparatively balanced distribution of true positives and true negatives.
        
        GPT-J-6B and Gemma-2B-IT also show improved balance compared to their zero-shot behavior.
        
        This demonstrates that decoder-only LLMs are capable of separating true and false statements, but only when ambiguity in their outputs is resolved heuristically.

2. Encoder–Decoder Models Remain Stable

        Models such as FLAN-T5-XL and FLAN-T5-XXL show similar performance patterns to Setup A, indicating that their behavior is less sensitive to post-processing heuristics.
        
        This suggests that encoder–decoder architectures may be intrinsically more robust for binary decision tasks, even in zero-shot or weakly constrained settings.

3. Encoder-Only Models Show Limited Gains

        BERT and RoBERTa models exhibit minor improvements but remain relatively weak overall.
        
        While their predictions are more balanced than those of large instruction-tuned LLMs, their lower accuracies suggest limited factual reasoning capacity without task-specific fine-tuning.

4. Performance Gains Come from Heuristics, Not Pure Reasoning

        The improvement observed in Setup B does not indicate that the models have learned fact-checking.
        
        Instead, it reflects:
        
        enforced binary decisions,
        
        resolution of ambiguous outputs,
        
        and the removal of abstentions (“UNK”).

When ambiguous outputs are resolved through heuristic post-processing, large language models exhibit substantially improved and more balanced performance. However, this improvement stems from forced decision-making rather than from genuine zero-shot fact-checking ability

# Testing Models with Subject

### Zero-Shot Model Evaluation (With Subject Context)
Same experimental setup as before, but now models receive the statement's subject category as additional context. This tests whether domain knowledge improves fact-checking performance.

## Setup A

In [ ]:
import re
import torch
import pandas as pd
from tqdm.auto import tqdm

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForMaskedLM,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
OUT_PATH = "1_Basics_Masterthesis/Results/statements_with_predictions_all_models_subjects.csv"

MODELS = {
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
    "deepseek-ai/deepseek-llm-7b-base": "causal",
    "EleutherAI/gpt-j-6B": "causal",
    "google/gemma-7b-it": "causal_chat_or_instruct",
    "google/gemma-2b-it": "causal_chat_or_instruct",
    "google/flan-t5-xl": "seq2seq",
    "google/flan-t5-xxl": "seq2seq",
    "t5-base": "seq2seq",
    "google/flan-t5-base": "seq2seq",
    "bert-base-uncased": "mlm",
    "bert-large-uncased": "mlm",
    "roberta-base": "mlm",
    "roberta-large": "mlm",
}

# -----------------------------
# Prompt 
# -----------------------------
def build_prompt(subjects: str, statement: str) -> str:
    subjects = "" if subjects is None else str(subjects)
    statement = "" if statement is None else str(statement)
    return (
        f"Subjects: {subjects}\n"
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )

# -----------------------------
# Robust: True/False aus Text
# -----------------------------
def normalize_tf(text: str):
    if text is None:
        return "UNK", None
    s = text.strip().lower()
    m = re.search(r"\b(true|false)\b", s)
    if not m:
        return "UNK", None
    return ("True", 1) if m.group(1) == "true" else ("False", 0)

# -----------------------------
# use Chat-Template if existing
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            return prompt
    return prompt

# -----------------------------
# MLM (BERT/RoBERTa): Mask-Scoring
# -----------------------------
def mlm_true_false(model, tokenizer, subjects: str, statement: str):
    if tokenizer.mask_token is None:
        return "UNK", None

    subjects = "" if subjects is None else str(subjects)
    statement = "" if statement is None else str(statement)

    prompt = (
        f"Subjects: {subjects}\n"
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: {tokenizer.mask_token}.\n"
        f"Answer: {tokenizer.mask_token}"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits  # [B, T, V]

    mask_id = tokenizer.mask_token_id
    mask_positions = (inputs["input_ids"][0] == mask_id).nonzero(as_tuple=False).flatten()
    if mask_positions.numel() == 0:
        return "UNK", None
    pos = mask_positions[-1].item()

    probs = torch.softmax(logits[0, pos], dim=-1)

    true_ids = tokenizer.encode(" true", add_special_tokens=False)
    false_ids = tokenizer.encode(" false", add_special_tokens=False)
    if len(true_ids) != 1:
        true_ids = tokenizer.encode("true", add_special_tokens=False)
    if len(false_ids) != 1:
        false_ids = tokenizer.encode("false", add_special_tokens=False)

    if len(true_ids) != 1 or len(false_ids) != 1:
        return "UNK", None

    p_true = probs[true_ids[0]].item()
    p_false = probs[false_ids[0]].item()

    return ("True", 1) if p_true >= p_false else ("False", 0)

# -----------------------------
# MAIN
# -----------------------------
def main():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    # check needed columns
    for col in ["statement", "subjects"]:
        if col not in df.columns:
            raise ValueError(f"Spalte '{col}' nicht gefunden. Spalten: {list(df.columns)}")

    statements = df["statement"].astype(str).tolist()
    subjects_list = df["subjects"].astype(str).tolist()

    for model_name in MODELS:
        short = model_name.split("/")[-1]
        df[f"pred_{short}"] = None
        df[f"pred_bin_{short}"] = None

    for model_name, mtype in MODELS.items():
        print(f"\n=== Lade: {model_name} ({mtype}) ===")
        short = model_name.split("/")[-1]

        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        if mtype in ("causal", "causal_chat_or_instruct"):
            if tokenizer.pad_token is None and tokenizer.eos_token is not None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, (subjects, statement) in enumerate(
                tqdm(list(zip(subjects_list, statements)), desc=f"{short} | generating", leave=True)
            ):
                prompt = build_prompt(subjects, statement)
                rendered = maybe_apply_chat_template(tokenizer, prompt, mtype)

                inputs = tokenizer(rendered, return_tensors="pt", truncation=True).to(device)

                output = model.generate(
                    **inputs,
                    max_new_tokens=6,
                    do_sample=False,
                    num_beams=1,
                    temperature=None,
                    top_p=None,
                    pad_token_id=tokenizer.eos_token_id
                )

                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                label_str, label_bin = normalize_tf(decoded)

                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        elif mtype == "seq2seq":
            model = AutoModelForSeq2SeqLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, (subjects, statement) in enumerate(
                tqdm(list(zip(subjects_list, statements)), desc=f"{short} | generating", leave=True)
            ):
                prompt = build_prompt(subjects, statement)

                inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

                output = model.generate(
                    **inputs,
                    max_new_tokens=6,
                    do_sample=False,
                    num_beams=1,
                    temperature=None,
                    top_p=None,
                    pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None
                )

                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                label_str, label_bin = normalize_tf(decoded)

                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        elif mtype == "mlm":
            model = AutoModelForMaskedLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, (subjects, statement) in enumerate(
                tqdm(list(zip(subjects_list, statements)), desc=f"{short} | scoring", leave=True)
            ):
                label_str, label_bin = mlm_true_false(model, tokenizer, subjects, statement)
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        else:
            raise ValueError(f"Unbekannter Modelltyp: {mtype}")

        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    df.to_csv(OUT_PATH, index=False)
    print(f"\nFertig. Gespeichert: {OUT_PATH}")

if __name__ == "__main__":
    main()



=== Lade: meta-llama/Llama-3.1-8B-Instruct (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | generating:   0%|          | 0/1267 [00:00<?, ?it/s]


=== Lade: deepseek-ai/deepseek-llm-7b-base (causal) ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...


=== Lade: roberta-base (mlm) ===


roberta-base | scoring:   0%|          | 0/1267 [00:00<?, ?it/s]


=== Lade: roberta-large (mlm) ===


roberta-large | scoring:   0%|          | 0/1267 [00:00<?, ?it/s]


Fertig. Gespeichert: 1_Basics_Masterthesis/Results/statements_with_predictions_all_models_subjects.csv


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/Results/statements_with_predictions_all_models_subjects.csv"
LABEL_COL = "label"
OUT_SUMMARY = "1_Basics_Masterthesis/Results/model_metrics_summary_subjects.csv"

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)

if LABEL_COL not in df.columns:
    raise ValueError(f"Label-Spalte '{LABEL_COL}' nicht gefunden!")

y_true = df[LABEL_COL].astype(int)

pred_cols = [c for c in df.columns if c.startswith("pred_bin_")]

print(f"Gefundene Modelle: {len(pred_cols)}")
print(pred_cols)

# -----------------------------
# EVALUATION
# -----------------------------
rows = []

for col in pred_cols:
    model_name = col.replace("pred_bin_", "")

    y_pred = df[col]

    mask = y_pred.notna()
    y_true_f = y_true[mask]
    y_pred_f = y_pred[mask].astype(int)

    acc = accuracy_score(y_true_f, y_pred_f)
    cm = confusion_matrix(y_true_f, y_pred_f, labels=[0, 1])

    print("\n" + "=" * 60)
    print(f"Model: {model_name}")
    print(f"Samples used: {len(y_true_f)}")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true_f, y_pred_f, digits=4))

    rows.append({
        "model": model_name,
        "accuracy": acc,
        "n_samples": len(y_true_f),
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    })

# -----------------------------
# SUMMARY TABLE
# -----------------------------
summary_df = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
summary_df.to_csv(OUT_SUMMARY, index=False)

print("\n==============================")
print("Summary saved to:")
print(OUT_SUMMARY)
print(summary_df)


Gefundene Modelle: 13
['pred_bin_Llama-3.1-8B-Instruct', 'pred_bin_deepseek-llm-7b-base', 'pred_bin_gpt-j-6B', 'pred_bin_gemma-7b-it', 'pred_bin_gemma-2b-it', 'pred_bin_flan-t5-xl', 'pred_bin_flan-t5-xxl', 'pred_bin_t5-base', 'pred_bin_flan-t5-base', 'pred_bin_bert-base-uncased', 'pred_bin_bert-large-uncased', 'pred_bin_roberta-base', 'pred_bin_roberta-large']

Model: Llama-3.1-8B-Instruct
Samples used: 1267
Accuracy: 0.5635
Confusion Matrix (rows=true, cols=pred):
[[  1 552]
 [  1 713]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5000    0.0018    0.0036       553
           1     0.5636    0.9986    0.7206       714

    accuracy                         0.5635      1267
   macro avg     0.5318    0.5002    0.3621      1267
weighted avg     0.5359    0.5635    0.4076      1267


Model: deepseek-llm-7b-base
Samples used: 1267
Accuracy: 0.5635
Confusion Matrix (rows=true, cols=pred):
[[  1 552]
 [  1 713]]

Classification Report:
   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Setup B

In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoModelForMaskedLM,
)

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
OUT_PATH = "1_Basics_Masterthesis/Results/statements_with_predictions_setupB_subjects.csv"

LABEL_COL = "label"

FALLBACK_LABEL_STR = "True"
FALLBACK_LABEL_BIN = 1 if FALLBACK_LABEL_STR == "True" else 0

TAKE_OCCURRENCE = "last"

MODELS = {
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
    "deepseek-ai/deepseek-llm-7b-base": "causal",
    "EleutherAI/gpt-j-6B": "causal",
    "google/gemma-7b-it": "causal_chat_or_instruct",
    "google/gemma-2b-it": "causal_chat_or_instruct",
    "google/flan-t5-xl": "seq2seq",
    "google/flan-t5-xxl": "seq2seq",
    "t5-base": "seq2seq",
    "google/flan-t5-base": "seq2seq",
    "bert-base-uncased": "mlm",
    "bert-large-uncased": "mlm",
    "roberta-base": "mlm",
    "roberta-large": "mlm",
}

# -----------------------------
# Prompt
# -----------------------------
def build_prompt(subjects: str, statement: str) -> str:
    subjects = "" if subjects is None else str(subjects)
    statement = "" if statement is None else str(statement)
    return (
        f"Subjects: {subjects}\n"
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )

# -----------------------------
# Setup B forced parser
# -----------------------------
def parse_true_false_forced(text: str):
    if text is None:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN

    s = str(text).strip().lower()
    matches = re.findall(r"\b(true|false)\b", s)

    if not matches:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN

    choice = matches[0] if TAKE_OCCURRENCE == "first" else matches[-1]
    return ("True", 1) if choice == "true" else ("False", 0)

# -----------------------------
# use Chat-Template if existing
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            return prompt
    return prompt

# -----------------------------
# MLM: Mask-Scoring (includes subjects)
# -----------------------------
def mlm_true_false(model, tokenizer, subjects: str, statement: str):
    if tokenizer.mask_token is None:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN

    subjects = "" if subjects is None else str(subjects)
    statement = "" if statement is None else str(statement)

    prompt = (
        f"Subjects: {subjects}\n"
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: {tokenizer.mask_token}.\n"
        f"Answer: {tokenizer.mask_token}"
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits

    mask_id = tokenizer.mask_token_id
    mask_positions = (inputs["input_ids"][0] == mask_id).nonzero(as_tuple=False).flatten()
    if mask_positions.numel() == 0:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN
    pos = mask_positions[-1].item()

    probs = torch.softmax(logits[0, pos], dim=-1)

    true_ids = tokenizer.encode(" true", add_special_tokens=False)
    false_ids = tokenizer.encode(" false", add_special_tokens=False)
    if len(true_ids) != 1:
        true_ids = tokenizer.encode("true", add_special_tokens=False)
    if len(false_ids) != 1:
        false_ids = tokenizer.encode("false", add_special_tokens=False)

    if len(true_ids) != 1 or len(false_ids) != 1:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN

    p_true = probs[true_ids[0]].item()
    p_false = probs[false_ids[0]].item()
    return ("True", 1) if p_true >= p_false else ("False", 0)

# -----------------------------
# MAIN
# -----------------------------
def run_predictions():
    df = pd.read_csv(DATA_PATH, sep=SEP)

    # Require these columns
    for col in ["statement", "subjects", LABEL_COL]:
        if col not in df.columns:
            raise ValueError(f"Spalte '{col}' nicht gefunden. Spalten: {list(df.columns)}")

    statements = df["statement"].astype(str).tolist()
    subjects_list = df["subjects"].astype(str).tolist()

    for model_name in MODELS:
        short = model_name.split("/")[-1]
        df[f"pred_{short}"] = None
        df[f"pred_bin_{short}"] = None
        df[f"raw_{short}"] = None

    for model_name, mtype in MODELS.items():
        print(f"\n=== Lade: {model_name} ({mtype}) ===")
        short = model_name.split("/")[-1]

        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        if mtype in ("causal", "causal_chat_or_instruct"):
            if tokenizer.pad_token is None and tokenizer.eos_token is not None:
                tokenizer.pad_token = tokenizer.eos_token

            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, (subjects, statement) in enumerate(
                tqdm(list(zip(subjects_list, statements)), desc=f"{short} | generating", leave=True)
            ):
                prompt = build_prompt(subjects, statement)
                rendered = maybe_apply_chat_template(tokenizer, prompt, mtype)

                inputs = tokenizer(rendered, return_tensors="pt", truncation=True).to(device)

                output = model.generate(
                    **inputs,
                    max_new_tokens=12,
                    do_sample=False,
                    num_beams=1,
                    pad_token_id=tokenizer.eos_token_id
                )

                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                label_str, label_bin = parse_true_false_forced(decoded)

                df.at[df.index[i], f"raw_{short}"] = decoded
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        elif mtype == "seq2seq":
            model = AutoModelForSeq2SeqLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, (subjects, statement) in enumerate(
                tqdm(list(zip(subjects_list, statements)), desc=f"{short} | generating", leave=True)
            ):
                prompt = build_prompt(subjects, statement)
                inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

                output = model.generate(
                    **inputs,
                    max_new_tokens=12,
                    do_sample=False,
                    num_beams=1,
                    pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None
                )

                decoded = tokenizer.decode(output[0], skip_special_tokens=True)
                label_str, label_bin = parse_true_false_forced(decoded)

                df.at[df.index[i], f"raw_{short}"] = decoded
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        elif mtype == "mlm":
            model = AutoModelForMaskedLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()

            for i, (subjects, statement) in enumerate(
                tqdm(list(zip(subjects_list, statements)), desc=f"{short} | scoring", leave=True)
            ):
                label_str, label_bin = mlm_true_false(model, tokenizer, subjects, statement)

                df.at[df.index[i], f"raw_{short}"] = None
                df.at[df.index[i], f"pred_{short}"] = label_str
                df.at[df.index[i], f"pred_bin_{short}"] = label_bin

        else:
            raise ValueError(f"Unbekannter Modelltyp: {mtype}")

        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    df.to_csv(OUT_PATH, index=False)
    print(f"\nPredictions gespeichert: {OUT_PATH}")
    return OUT_PATH

# -----------------------------
# RUN
# -----------------------------
if __name__ == "__main__":
    pred_path = run_predictions()



=== Lade: meta-llama/Llama-3.1-8B-Instruct (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | generating:   0%|          | 0/1267 [00:00<?, ?it/s]


=== Lade: deepseek-ai/deepseek-llm-7b-base (causal) ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...


=== Lade: roberta-base (mlm) ===


roberta-base | scoring:   0%|          | 0/1267 [00:00<?, ?it/s]


=== Lade: roberta-large (mlm) ===


roberta-large | scoring:   0%|          | 0/1267 [00:00<?, ?it/s]


Predictions gespeichert: 1_Basics_Masterthesis/Results/statements_with_predictions_setupB_subjects.csv


In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/Results/statements_with_predictions_setupB_subjects.csv"
LABEL_COL = "label"
OUT_SUMMARY = "1_Basics_Masterthesis/Results/model_metrics_summary_setupB_subjects.csv"

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv(DATA_PATH)

if LABEL_COL not in df.columns:
    raise ValueError(f"Label-Spalte '{LABEL_COL}' nicht gefunden!")

y_true = df[LABEL_COL].astype(int)

pred_cols = [c for c in df.columns if c.startswith("pred_bin_")]

print(f"Gefundene Modelle: {len(pred_cols)}")
print(pred_cols)

# -----------------------------
# EVALUATION
# -----------------------------
rows = []

for col in pred_cols:
    model_name = col.replace("pred_bin_", "")

    y_pred = df[col]

    mask = y_pred.notna()
    y_true_f = y_true[mask]
    y_pred_f = y_pred[mask].astype(int)

    acc = accuracy_score(y_true_f, y_pred_f)
    cm = confusion_matrix(y_true_f, y_pred_f, labels=[0, 1])

    print("\n" + "=" * 60)
    print(f"Model: {model_name}")
    print(f"Samples used: {len(y_true_f)}")
    print(f"Accuracy: {acc:.4f}")
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm)
    print("\nClassification Report:")
    print(classification_report(y_true_f, y_pred_f, digits=4))

    rows.append({
        "model": model_name,
        "accuracy": acc,
        "n_samples": len(y_true_f),
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    })

# -----------------------------
# SUMMARY TABLE
# -----------------------------
summary_df = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
summary_df.to_csv(OUT_SUMMARY, index=False)

print("\n==============================")
print("Summary saved to:")
print(OUT_SUMMARY)
print(summary_df)


Gefundene Modelle: 13
['pred_bin_Llama-3.1-8B-Instruct', 'pred_bin_deepseek-llm-7b-base', 'pred_bin_gpt-j-6B', 'pred_bin_gemma-7b-it', 'pred_bin_gemma-2b-it', 'pred_bin_flan-t5-xl', 'pred_bin_flan-t5-xxl', 'pred_bin_t5-base', 'pred_bin_flan-t5-base', 'pred_bin_bert-base-uncased', 'pred_bin_bert-large-uncased', 'pred_bin_roberta-base', 'pred_bin_roberta-large']

Model: Llama-3.1-8B-Instruct
Samples used: 1267
Accuracy: 0.5683
Confusion Matrix (rows=true, cols=pred):
[[381 172]
 [375 339]]

Classification Report:
              precision    recall  f1-score   support

           0     0.5040    0.6890    0.5821       553
           1     0.6634    0.4748    0.5535       714

    accuracy                         0.5683      1267
   macro avg     0.5837    0.5819    0.5678      1267
weighted avg     0.5938    0.5683    0.5660      1267


Model: deepseek-llm-7b-base
Samples used: 1267
Accuracy: 0.4373
Confusion Matrix (rows=true, cols=pred):
[[553   0]
 [713   1]]

Classification Report:
   

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Results of first tests

## Setup A without subjects

| Model                 | Accuracy | Macro-F1 | Balanced Acc |  TN |  FP |  FN |  TP |
| --------------------- | -------: | -------: | -----------: | --: | --: | --: | --: |
| flan-t5-xl            |    0.579 |    0.571 |        0.570 | 280 | 273 | 261 | 453 |
| bert-base-uncased     |    0.573 |    0.423 |        0.517 |  40 | 513 |  28 | 685 |
| Llama-3.1-8B-Instruct |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| deepseek-llm-7b-base  |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| gpt-j-6B              |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| gemma-7b-it           |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| gemma-2b-it           |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| bert-large-uncased    |    0.563 |    0.360 |        0.500 |   0 | 553 |   0 | 713 |
| t5-base               |    0.555 |    0.400 |        0.499 |  29 | 523 |  38 | 672 |
| flan-t5-xxl           |    0.546 |    0.538 |        0.572 | 430 | 123 | 452 | 261 |
| roberta-base          |    0.523 |    0.506 |        0.507 | 213 | 340 | 264 | 449 |
| flan-t5-base          |    0.522 |    0.522 |        0.535 | 351 | 202 | 403 | 310 |
| roberta-large         |    0.474 |    0.471 |        0.492 | 349 | 204 | 462 | 251 |


## Setup B without Subjects

| Model                 | Accuracy | Macro-F1 | Balanced Acc |  TN |  FP |  FN |  TP |
| --------------------- | -------: | -------: | -----------: | --: | --: | --: | --: |
| gemma-7b-it           |    0.612 |    0.583 |        0.588 | 222 | 331 | 161 | 553 |
| flan-t5-xl            |    0.579 |    0.571 |        0.570 | 280 | 273 | 261 | 453 |
| bert-base-uncased     |    0.573 |    0.423 |        0.517 |  40 | 513 |  28 | 686 |
| bert-large-uncased    |    0.564 |    0.360 |        0.500 |   0 | 553 |   0 | 714 |
| gpt-j-6B              |    0.562 |    0.363 |        0.499 |   2 | 551 |   4 | 710 |
| gemma-2b-it           |    0.561 |    0.489 |        0.522 | 117 | 436 | 120 | 594 |
| t5-base               |    0.549 |    0.420 |        0.497 |  49 | 504 |  68 | 646 |
| flan-t5-xxl           |    0.546 |    0.538 |        0.572 | 430 | 123 | 452 | 262 |
| Llama-3.1-8B-Instruct |    0.529 |    0.514 |        0.560 | 445 | 108 | 489 | 225 |
| roberta-base          |    0.523 |    0.506 |        0.508 | 213 | 340 | 264 | 450 |
| flan-t5-base          |    0.522 |    0.522 |        0.535 | 351 | 202 | 403 | 311 |
| roberta-large         |    0.474 |    0.471 |        0.492 | 349 | 204 | 462 | 252 |
| deepseek-llm-7b-base  |    0.436 |    0.305 |        0.500 | 552 |   1 | 713 |   1 |


## Setup A with subjects

| Model                 | Accuracy | Macro-F1 | Balanced Acc |  TN |  FP |  FN |  TP |
| --------------------- | -------: | -------: | -----------: | --: | --: | --: | --: |
| flan-t5-xl            |    0.571 |    0.570 |        0.574 | 334 | 219 | 325 | 389 |
| bert-base-uncased     |    0.567 |    0.383 |        0.506 |  13 | 540 |   8 | 705 |
| Llama-3.1-8B-Instruct |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| deepseek-llm-7b-base  |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| gpt-j-6B              |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| gemma-7b-it           |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| gemma-2b-it           |    0.564 |    0.362 |        0.500 |   1 | 552 |   1 | 713 |
| bert-large-uncased    |    0.563 |    0.360 |        0.500 |   0 | 553 |   0 | 713 |
| t5-base               |    0.555 |    0.369 |        0.495 |   8 | 545 |  18 | 694 |
| flan-t5-xxl           |    0.537 |    0.525 |        0.566 | 440 | 113 | 473 | 240 |
| flan-t5-base          |    0.509 |    0.503 |        0.530 | 387 | 166 | 456 | 257 |
| roberta-large         |    0.449 |    0.375 |        0.501 | 503 |  50 | 647 |  66 |
| roberta-base          |    0.445 |    0.379 |        0.495 | 489 |  64 | 638 |  75 |


## Setup B with subjects

| Model                 | Accuracy | Macro-F1 | Balanced Acc |  TN |  FP |  FN |  TP |
| --------------------- | -------: | -------: | -----------: | --: | --: | --: | --: |
| gemma-7b-it           |    0.608 |    0.568 |        0.578 | 192 | 361 | 136 | 578 |
| flan-t5-xl            |    0.571 |    0.570 |        0.574 | 334 | 219 | 325 | 389 |
| Llama-3.1-8B-Instruct |    0.568 |    0.568 |        0.582 | 381 | 172 | 375 | 339 |
| bert-base-uncased     |    0.567 |    0.383 |        0.506 |  13 | 540 |   8 | 706 |
| bert-large-uncased    |    0.564 |    0.360 |        0.500 |   0 | 553 |   0 | 714 |
| gemma-2b-it           |    0.563 |    0.480 |        0.520 | 104 | 449 | 105 | 609 |
| gpt-j-6B              |    0.556 |    0.362 |        0.494 |   3 | 550 |  12 | 702 |
| flan-t5-xxl           |    0.537 |    0.526 |        0.567 | 440 | 113 | 473 | 241 |
| flan-t5-base          |    0.509 |    0.504 |        0.531 | 387 | 166 | 456 | 258 |
| t5-base               |    0.493 |    0.459 |        0.469 | 152 | 401 | 241 | 473 |
| roberta-large         |    0.450 |    0.376 |        0.502 | 503 |  50 | 647 |  67 |
| roberta-base          |    0.446 |    0.380 |        0.495 | 489 |  64 | 638 |  76 |
| deepseek-llm-7b-base  |    0.437 |    0.305 |        0.501 | 553 |   0 | 713 |   1 |


## Overall Results

| Model                     | Setup A<br>No Subjects | Setup B<br>No Subjects | Setup A<br>With Subjects | Setup B<br>With Subjects |
|---------------------------|------------------------|------------------------|--------------------------|--------------------------|
| Llama-3.1-8B-Instruct     | 0.5635                 | 0.5288                 | 0.5635                   | 0.5683                   |
| deepseek-llm-7b-base      | 0.5635                 | 0.4365                 | 0.5635                   | 0.4373                   |
| gpt-j-6B                  | 0.5635                 | 0.5620                 | 0.5635                   | 0.5564                   |
| gemma-7b-it               | 0.5635                 | **0.6117**             | 0.5635                   | **0.6077**               |
| gemma-2b-it               | 0.5635                 | 0.5612                 | 0.5635                   | 0.5627                   |
| flan-t5-xl                | **0.5785**             | **0.5785**             | **0.5706**               | **0.5706**               |
| flan-t5-xxl               | 0.5458                 | 0.5462                 | 0.5371                   | 0.5375                   |
| t5-base                   | 0.5555                 | 0.5485                 | 0.5549                   | 0.4933                   |
| flan-t5-base              | 0.5221                 | 0.5225                 | 0.5087                   | 0.5091                   |
| bert-base-uncased         | 0.5727                 | 0.5730                 | 0.5671                   | 0.5675                   |
| bert-large-uncased        | 0.5632                 | 0.5635                 | 0.5632                   | 0.5635                   |
| roberta-base              | 0.5229                 | 0.5233                 | 0.4455                   | 0.4459                   |
| roberta-large             | 0.4739                 | 0.4743                 | 0.4494                   | 0.4499                   |


## Conclusion

Across all baseline experiments, several consistent patterns emerge. First, Setup A without forced decision parsing leads to degenerate model behavior for most decoder-only and encoder-only models, which overwhelmingly predict the majority class (“True”). This results in superficially acceptable accuracy values around 56%, but extremely poor recall for the minority class, indicating that these models fail to meaningfully engage with the task under strict output filtering.

In contrast, Setup B (forced binary parsing) produces more informative prediction distributions. While overall accuracy sometimes decreases, confusion matrices reveal more balanced decision behavior, particularly for instruction-tuned models such as Gemma-7B-IT and LLaMA-3.1-8B-Instruct. This demonstrates that evaluation methodology has a substantial impact on apparent model performance and should be considered carefully in prompt-based fact-checking tasks.

The inclusion of subject information has no effect in Setup A, as model outputs remain dominated by single-class predictions. However, in Setup B, subject-aware prompts lead to modest but consistent performance improvements for several models, most notably Gemma-7B-IT and LLaMA-3.1-8B-Instruct, suggesting that domain cues are only beneficial when models are encouraged to make explicit binary decisions.

Overall, instruction-tuned decoder-only models outperform encoder-only baselines and encoder–decoder models under comparable conditions, particularly when evaluated with forced decision parsing.

# Prompt-Engineering

Tests different prompt formulations and complexity levels to optimize model performance. Includes experiments with chain-of-thought reasoning and self-consistency techniques.


In [12]:
import os
import re
import torch
import pandas as pd
from tqdm.auto import tqdm
import torch.nn.functional as F
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
)

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
N_ROWS = None
LABEL_COL = "label"
STATEMENT_COL = "statement"
SUBJECTS_COL = "subjects"

OUT_PRED_PATH = "1_Basics_Masterthesis/Results/prompt_eng_setupB_predictions.csv"
OUT_SUMMARY_PATH = "1_Basics_Masterthesis/Results/prompt_eng_setupB_summary.csv"

# Setup B forced parsing
FALLBACK_LABEL_STR = "True"
FALLBACK_LABEL_BIN = 1 if FALLBACK_LABEL_STR == "True" else 0
TAKE_OCCURRENCE = "last"

# -----------------------------
# MODELS (4 models)
# -----------------------------
MODELS = {
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
    "deepseek-ai/deepseek-llm-7b-base": "causal",
    "google/gemma-7b-it": "causal_chat_or_instruct",
    "google/flan-t5-xl": "seq2seq",
}

# -----------------------------
# HELPERS
# -----------------------------
def parse_true_false_forced(text: str):
    if text is None:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN
    s = str(text).strip().lower()
    matches = re.findall(r"\b(true|false)\b", s)
    if not matches:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN
    choice = matches[0] if TAKE_OCCURRENCE == "first" else matches[-1]
    return ("True", 1) if choice == "true" else ("False", 0)

def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

def subjects_or_empty(subjects: str, use_subjects: bool) -> str:
    return str(subjects) if use_subjects else ""

def get_label_token_id(tokenizer, label: str):
    """
    Try to find a single token id for the label (True/False).
    We try common variants with/without leading space and casing.
    Returns token_id or None if it is not single-token.
    """
    candidates = [label, label.lower(), " " + label, " " + label.lower()]
    for c in candidates:
        ids = tokenizer.encode(c, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return None

def confidence_causal_next_token(model, tokenizer, rendered_prompt: str):
    """
    Returns (p_true, p_false) based on next-token distribution after the prompt.
    If True/False are not single-token for this tokenizer, returns (None, None).
    """
    true_id = get_label_token_id(tokenizer, "True")
    false_id = get_label_token_id(tokenizer, "False")
    if true_id is None or false_id is None:
        return None, None

    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits  # [B, T, V]
    next_logits = logits[0, -1, :]      # next token after prompt
    probs = F.softmax(next_logits, dim=-1)

    p_true = probs[true_id].item()
    p_false = probs[false_id].item()
    return p_true, p_false

def confidence_seq2seq_first_token(model, tokenizer, prompt: str):
    """
    Returns (p_true, p_false) from the first decoder step.
    If True/False are not single-token, returns (None, None).
    """
    true_id = get_label_token_id(tokenizer, "True")
    false_id = get_label_token_id(tokenizer, "False")
    if true_id is None or false_id is None:
        return None, None

    enc = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    # For T5-family, decoder_start_token_id is usually set; fallback to pad_token_id
    start_id = model.config.decoder_start_token_id
    if start_id is None:
        start_id = tokenizer.pad_token_id

    decoder_input_ids = torch.tensor([[start_id]], device=device)

    with torch.no_grad():
        out = model(
            input_ids=enc["input_ids"],
            attention_mask=enc.get("attention_mask", None),
            decoder_input_ids=decoder_input_ids
        )
        logits = out.logits  # [B, 1, V] first decoder token distribution

    probs = F.softmax(logits[0, 0, :], dim=-1)
    p_true = probs[true_id].item()
    p_false = probs[false_id].item()
    return p_true, p_false


# -----------------------------
# PROMPT VARIANTS
# Each prompt is a function(statement, subjects, use_subjects) -> string
# run EVERY prompt in both modes (use_subjects=True/False).
# -----------------------------
def p0_basic(statement, subjects, use_subjects):
    sb = subjects_or_empty(subjects, use_subjects)
    if use_subjects:
        return (
            f"Subjects: {sb}\n"
            f"Statement: {statement}\n"
            f"Is this statement true or false?\n"
            f"Answer with only one word: True or False.\n"
            f"Answer:"
        )
    else:
        return (
            f"Statement: {statement}\n"
            f"Is this statement true or false?\n"
            f"Answer with only one word: True or False.\n"
            f"Answer:"
        )

def p1_fake_news_no_guess(statement, subjects, use_subjects):
    sb = subjects_or_empty(subjects, use_subjects)
    base = (
        f"Statement: {statement}\n"
        f"Task: Determine whether the statement is factually correct or fake news. "
        f"Use only publicly verifiable and objective facts. Do not guess.\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )
    if use_subjects:
        return f"Subjects: {sb}\n" + base
    return base

def p2_one_shot_example(statement, subjects, use_subjects):
    sb = subjects_or_empty(subjects, use_subjects)
    prompt = (
        f"Task: Decide if the statement is factually correct.\n"
        f"Answer only with True or False.\n"
        f"Example:\n"
        f"Statement: The sky is green.\n"
        f"Answer: False\n"
        f"Now evaluate:\n"
        f"Statement: {statement}\n"
        f"Answer:"
    )
    if use_subjects:
        return f"Subjects: {sb}\n" + prompt
    return prompt

def p3_factcheck_assistant(statement, subjects, use_subjects):
    sb = subjects_or_empty(subjects, use_subjects)
    prompt = (
        f"You are an AI assistant specialized in fact-checking.\n"
        f"Task: Determine whether each statement is factually correct based on publicly verifiable information.\n"
        f"Respond strictly with one word: True or False.\n\n"
        f"Example:\n"
        f"Statement: The sky is green.\n"
        f"Answer: False\n\n"
        f"Statement: {statement}\n"
        f"Answer:"
    )
    if use_subjects:
        return f"Subjects: {sb}\n" + prompt
    return prompt

def p4_subjects_as_context_clues(statement, subjects, use_subjects):
    sb = subjects_or_empty(subjects, use_subjects)
    if use_subjects:
        return (
            f"You are an AI assistant specialized in fact-checking factual statements.\n"
            f"Your task is to determine whether the following statement is factually correct.\n"
            f"You may use the following subjects as context clues: {sb}.\n"
            f"Respond strictly with one word: True or False.\n\n"
            f"Example:\n"
            f"Statement: The sky is green.\n"
            f"Answer: False\n\n"
            f"Statement: {statement}\n"
            f"Answer:"
        )
    else:
        return (
            f"You are an AI assistant specialized in fact-checking factual statements.\n"
            f"Your task is to determine whether the following statement is factually correct.\n"
            f"Respond strictly with one word: True or False.\n\n"
            f"Example:\n"
            f"Statement: The sky is green.\n"
            f"Answer: False\n\n"
            f"Statement: {statement}\n"
            f"Answer:"
        )

PROMPTS = [
    ("P0_basic", p0_basic),
    ("P1_fake_news_no_guess", p1_fake_news_no_guess),
    ("P2_one_shot_example", p2_one_shot_example),
    ("P3_factcheck_assistant", p3_factcheck_assistant),
    ("P4_subjects_as_context_clues", p4_subjects_as_context_clues),
]

# -----------------------------
# GENERATION
# -----------------------------
def generate_causal(model, tokenizer, rendered_prompt: str):
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=12,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def generate_seq2seq(model, tokenizer, prompt: str):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=12,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.eos_token_id if tokenizer.eos_token_id is not None else None
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# -----------------------------
# MAIN RUN
# -----------------------------
def run():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    for col in [STATEMENT_COL, SUBJECTS_COL, LABEL_COL]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}'. Found: {list(df.columns)}")

    df[STATEMENT_COL] = df[STATEMENT_COL].astype(str)
    df[SUBJECTS_COL] = df[SUBJECTS_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    rows = []
    summary_rows = []

    for model_name, model_type in MODELS.items():
        print(f"\n=== Loading: {model_name} ({model_type}) ===")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        if model_type in ("causal", "causal_chat_or_instruct"):
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()
        elif model_type == "seq2seq":
            model = AutoModelForSeq2SeqLM.from_pretrained(
                model_name,
                dtype=torch.float16 if device.type == "cuda" else None,
            ).to(device)
            model.eval()
        else:
            raise ValueError(f"Unknown model type: {model_type}")

        for prompt_name, prompt_fn in PROMPTS:
            for use_subjects in [False, True]:
                mode = "with_subjects" if use_subjects else "no_subjects"
                desc = f"{model_name.split('/')[-1]} | {prompt_name} | {mode}"
                iterator = tqdm(df.itertuples(index=True), total=len(df), desc=desc, leave=True)

                preds = []
                trues = []

                for r in iterator:
                    statement = getattr(r, STATEMENT_COL)
                    subjects = getattr(r, SUBJECTS_COL)
                    y_true = getattr(r, LABEL_COL)

                    prompt = prompt_fn(statement, subjects, use_subjects)

                    if model_type in ("causal", "causal_chat_or_instruct"):
                        rendered = maybe_apply_chat_template(tokenizer, prompt, model_type)
                        raw = generate_causal(model, tokenizer, rendered)
                    else:
                        raw = generate_seq2seq(model, tokenizer, prompt)

                    pred_str, pred_bin = parse_true_false_forced(raw)

                    # confidence from logits (Setup B)
                    if model_type in ("causal", "causal_chat_or_instruct"):
                        p_true, p_false = confidence_causal_next_token(model, tokenizer, rendered)
                    else:
                        p_true, p_false = confidence_seq2seq_first_token(model, tokenizer, prompt)
                    
                    if p_true is None or p_false is None:
                        conf = None
                    else:
                        conf = p_true if pred_bin == 1 else p_false


                    rows.append({
                        "idx": r.Index,
                        "model": model_name,
                        "model_short": model_name.split("/")[-1],
                        "prompt": prompt_name,
                        "mode": mode,
                        "use_subjects": use_subjects,
                        "y_true": y_true,
                        "y_pred": pred_bin,
                        "y_pred_str": pred_str,
                        "raw": raw,
                        "p_true": p_true,
                        "p_false": p_false,
                        "confidence": conf,
                    })

                    preds.append(pred_bin)
                    trues.append(y_true)

                # metrics per (model, prompt, mode)
                acc = accuracy_score(trues, preds)
                macro_f1 = f1_score(trues, preds, average="macro")
                cm = confusion_matrix(trues, preds, labels=[0, 1])  # rows true, cols pred
                tn, fp, fn, tp = cm.ravel()

                conf_arr = np.array([x for x in [row["confidence"] for row in rows[-len(trues):]] if x is not None], dtype=float)

                mean_conf = float(np.mean(conf_arr)) if conf_arr.size > 0 else None
                
                chunk = rows[-len(trues):]
                conf_correct = [c["confidence"] for c in chunk if c["confidence"] is not None and c["y_pred"] == c["y_true"]]
                conf_wrong   = [c["confidence"] for c in chunk if c["confidence"] is not None and c["y_pred"] != c["y_true"]]
                mean_conf_correct = float(np.mean(conf_correct)) if len(conf_correct) > 0 else None
                mean_conf_wrong   = float(np.mean(conf_wrong)) if len(conf_wrong) > 0 else None


                summary_rows.append({
                    "model": model_name,
                    "model_short": model_name.split("/")[-1],
                    "prompt": prompt_name,
                    "mode": mode,
                    "accuracy": acc,
                    "macro_f1": macro_f1,
                    "n": len(trues),
                    "tn": tn, "fp": fp, "fn": fn, "tp": tp,
                    "mean_confidence": mean_conf,
                    "mean_conf_correct": mean_conf_correct,
                    "mean_conf_wrong": mean_conf_wrong,

                })

        # cleanup
        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(OUT_PRED_PATH, index=False)

    summary_df = pd.DataFrame(summary_rows).sort_values(["accuracy", "macro_f1"], ascending=False)
    summary_df.to_csv(OUT_SUMMARY_PATH, index=False)

    print(f"\nSaved predictions: {OUT_PRED_PATH}")
    print(f"Saved summary: {OUT_SUMMARY_PATH}")

    # Print a markdown table (top 30 rows by accuracy, then macro-f1)
    top = summary_df.head(30).copy()
    print("\nTop results (Markdown):\n")
    print(top.to_markdown(index=False))

if __name__ == "__main__":
    run()



=== Loading: meta-llama/Llama-3.1-8B-Instruct (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | P0_basic | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | P0_basic | with_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | P1_fake_news_no_guess | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

...

flan-t5-xl | P3_factcheck_assistant | with_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

flan-t5-xl | P4_subjects_as_context_clues | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

flan-t5-xl | P4_subjects_as_context_clues | with_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]


Saved predictions: 1_Basics_Masterthesis/Results/prompt_eng_setupB_predictions.csv
Saved summary: 1_Basics_Masterthesis/Results/prompt_eng_setupB_summary.csv

Top results (Markdown):



ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.

In [16]:
import pandas as pd

# -----------------------------
# LOAD SUMMARY
# -----------------------------
SUMMARY_PATH = "1_Basics_Masterthesis/Results/prompt_eng_setupB_summary.csv"

df = pd.read_csv(SUMMARY_PATH)

print(f"Loaded {len(df)} experiment rows")
df



Loaded 40 experiment rows


,model,model_short,prompt,mode,accuracy,macro_f1,n,tn,fp,fn,tp,mean_confidence,mean_conf_correct,mean_conf_wrong
0,google/gemma-7b-it,gemma-7b-it,P4_subjects_as_context_clues,with_subjects,0.614838,0.586258,1267,223,330,158,556,0.913033,0.922152,0.898477
1,google/gemma-7b-it,gemma-7b-it,P0_basic,no_subjects,0.611681,0.583237,1267,222,331,161,553,0.959342,0.965208,0.950100
2,google/gemma-7b-it,gemma-7b-it,P1_fake_news_no_guess,with_subjects,0.608524,0.568700,1267,193,360,136,578,0.923257,0.932908,0.908256
3,google/gemma-7b-it,gemma-7b-it,P0_basic,with_subjects,0.607735,0.567601,1267,192,361,136,578,0.940097,0.950858,0.923424
4,google/gemma-7b-it,gemma-7b-it,P1_fake_news_no_guess,no_subjects,0.606946,0.573460,1267,207,346,152,562,0.953124,0.962548,0.938573
5,google/gemma-7b-it,gemma-7b-it,P4_subjects_as_context_clues,no_subjects,0.599053,0.591072,1267,291,262,246,468,0.945926,0.955262,0.931978
6,google/gemma-7b-it,gemma-7b-it,P3_factcheck_assistant,with_subjects,0.595896,0.592176,1267,317,236,276,438,0.846679,0.869019,0.813737
7,google/flan-t5-xl,flan-t5-xl,P1_fake_news_no_guess,with_subjects,0.595107,0.539003,1267,156,397,116,598,0.562354,0.579754,0.536781
8,google/gemma-7b-it,gemma-7b-it,P2_one_shot_example,no_subjects,0.594317,0.578559,1267,254,299,215,499,0.332829,0.364237,0.286815
9,google/gemma-7b-it,gemma-7b-it,P2_one_shot_example,with_subjects,0.593528,0.581592,1267,269,284,231,483,0.315404,0.333795,0.288550


In [17]:
df_sorted = df.sort_values("accuracy", ascending=False)

df_sorted[
    ["model_short", "prompt", "mode", "accuracy", "macro_f1"]
].head(15)


,model_short,prompt,mode,accuracy,macro_f1
0,gemma-7b-it,P4_subjects_as_context_clues,with_subjects,0.614838,0.586258
1,gemma-7b-it,P0_basic,no_subjects,0.611681,0.583237
2,gemma-7b-it,P1_fake_news_no_guess,with_subjects,0.608524,0.568700
3,gemma-7b-it,P0_basic,with_subjects,0.607735,0.567601
4,gemma-7b-it,P1_fake_news_no_guess,no_subjects,0.606946,0.573460
5,gemma-7b-it,P4_subjects_as_context_clues,no_subjects,0.599053,0.591072
6,gemma-7b-it,P3_factcheck_assistant,with_subjects,0.595896,0.592176
7,flan-t5-xl,P1_fake_news_no_guess,with_subjects,0.595107,0.539003
8,gemma-7b-it,P2_one_shot_example,no_subjects,0.594317,0.578559
9,gemma-7b-it,P2_one_shot_example,with_subjects,0.593528,0.581592


In [18]:
best_per_model = (
    df
    .sort_values(["model_short", "accuracy", "macro_f1"], ascending=False)
    .groupby("model_short")
    .head(1)
)

best_per_model[
    ["model_short", "prompt", "mode", "accuracy", "macro_f1"]
]


,model_short,prompt,mode,accuracy,macro_f1
0,gemma-7b-it,P4_subjects_as_context_clues,with_subjects,0.614838,0.586258
7,flan-t5-xl,P1_fake_news_no_guess,with_subjects,0.595107,0.539003
19,deepseek-llm-7b-base,P2_one_shot_example,with_subjects,0.578532,0.418280
17,Llama-3.1-8B-Instruct,P1_fake_news_no_guess,with_subjects,0.578532,0.578511


In [19]:
subjects_comparison = (
    df
    .groupby(["model_short", "mode"])
    .agg(
        accuracy_mean=("accuracy", "mean"),
        f1_mean=("macro_f1", "mean"),
        n=("accuracy", "count")
    )
    .reset_index()
)

subjects_comparison


,model_short,mode,accuracy_mean,f1_mean,n
0,Llama-3.1-8B-Instruct,no_subjects,0.534807,0.525237,5
1,Llama-3.1-8B-Instruct,with_subjects,0.554223,0.548050,5
2,deepseek-llm-7b-base,no_subjects,0.532597,0.415763,5
3,deepseek-llm-7b-base,with_subjects,0.532439,0.391947,5
4,flan-t5-xl,no_subjects,0.582794,0.556609,5
5,flan-t5-xl,with_subjects,0.586740,0.549909,5
6,gemma-7b-it,no_subjects,0.600631,0.583240,5
7,gemma-7b-it,with_subjects,0.604104,0.579265,5


In [20]:
subjects_pivot = subjects_comparison.pivot(
    index="model_short",
    columns="mode",
    values="accuracy_mean"
)

subjects_pivot


mode,no_subjects,with_subjects
model_short,,
Llama-3.1-8B-Instruct,0.534807,0.554223
deepseek-llm-7b-base,0.532597,0.532439
flan-t5-xl,0.582794,0.586740
gemma-7b-it,0.600631,0.604104


## Conclusion

Across all evaluated models, prompt design has a measurable but model-dependent impact on fact-checking performance. The strongest overall results were achieved by Gemma-7B-IT, which consistently outperformed the other models under multiple prompt formulations, reaching a peak accuracy of ≈61.5% and a macro-F1 of ≈0.59. This indicates that Gemma benefits particularly well from explicit task framing and contextual guidance.

The inclusion of subjects generally led to small but consistent performance gains for instruction-tuned decoder models such as Gemma-7B-IT and Llama-3.1-8B-Instruct, especially when subjects were presented as contextual clues rather than direct indicators of correctness. In contrast, encoder-decoder models like FLAN-T5-XL showed relatively stable performance across prompts, suggesting a lower sensitivity to prompt phrasing and subject information.

Prompt variants that included explicit role assignment (e.g., “AI assistant specialized in fact-checking”) and clear output constraints (“Respond strictly with one word: True or False”) consistently ranked among the best-performing prompts. One-shot examples provided moderate improvements in some cases but did not universally outperform simpler task-oriented prompts.

Confidence scores further revealed that higher confidence does not necessarily imply higher correctness, particularly for models such as DeepSeek-7B, which exhibited extremely low confidence variance despite poor classification performance. In contrast, Gemma-7B-IT demonstrated both higher confidence and better alignment between confidence and correctness, making it a strong candidate for downstream techniques such as self-consistency and iterative refinement.

Overall, these results show that prompt engineering and subject inclusion can meaningfully improve zero-shot fact-checking, but the magnitude of improvement depends strongly on the underlying model architecture and instruction tuning. Based on these findings, subsequent experiments focusing on Gemma-7B-IT and Llama-3.1-8B-Instruct are well-justified.

1) Gemma-7B-IT is the clear winner in this prompt-engineering phase.
Across many prompts it stays at the top, peaking at ~0.615 accuracy / ~0.586 macro-F1 (P4 with subjects). It also performs strongly even without subjects (P0 no-subjects ~0.612).
Interpretation: Gemma is both prompt-sensitive (can improve with better instruction) and stable (doesn’t collapse).

2) Subjects help — but only for some models and prompts.
For Gemma, adding subjects sometimes improves the best case (P4 with subjects is best overall), but not universally (P0 no-subjects is also very strong).
For LLaMA, subjects clearly help in the best prompt (P1 with subjects gives the best LLaMA result ~0.579 acc / ~0.579 macro-F1), while some other prompts degrade.

3) LLaMA shows strong prompt dependence and inconsistent behavior.
Its performance ranges widely (roughly 0.507 → 0.579 accuracy depending on prompt/mode).
This is actually a good sign for your thesis: it proves that prompt design is a major factor and motivates reprompting/self-refinement later.

4) FLAN-T5-XL improves with instruction, but confidence is low and it behaves differently than decoder models.
Its best results are around ~0.595 accuracy in your grid (P1 with subjects), but the confidence scores are much smaller and the effect of subjects is not consistently positive.
Interpretation: Encoder-decoder models are often more “task-following” but less “confident” in the same logprob sense, so confidence comparisons must be done carefully across architectures.

5) DeepSeek-7B-Base is unreliable here (likely incompatible with the instruction format / parsing).
It shows extreme patterns and tiny confidence values, and sometimes near-trivial behavior.
Interpretation: You should treat DeepSeek-base as a base model that likely needs either:

an instruct version, or

stronger prompting / constrained decoding, or

a different scoring method (e.g., logit comparison for true vs false).

6) Confidence is informative within a model, but not directly comparable across models.
Gemma and LLaMA show high mean confidence (~0.75–0.96) whereas FLAN and DeepSeek show very low confidence.
This can happen because:

different tokenization,

seq2seq vs causal generation,

different output lengths and EOS behavior.
Use confidence mainly for: calibration curves and “confidence correct vs wrong” within the same model.

# Advanced Reasoning Techniques
This section experiments with more sophisticated prompting strategies including:
- **Chain-of-Thought (CoT)**: Instructing models to reason step-by-step
- **Self-Consistency**: Sampling multiple outputs and selecting the most common response
- **Iterative Refinement**: Refining predictions through multi-turn interactions

## Selfconsistency and Reprompting on Gemma and Llama

In [ ]:
import os
import re
import math
import random
import torch
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
N_ROWS = None

LABEL_COL = "label" 
STATEMENT_COL = "statement"
SUBJECTS_COL = "subjects"

OUT_PRED_PATH = "1_Basics_Masterthesis/Results/reprompt_vs_selfcons_predictions.csv"
OUT_SUMMARY_PATH = "1_Basics_Masterthesis/Results/reprompt_vs_selfcons_summary.csv"

# Methods settings
SEED = 42
random.seed(SEED)

# Self-consistency
N_VARIANTS = 5
WEIGHTED_VOTE = True

# Reprompting threshold
CONFIDENCE_THRESHOLD = 0.75

# Candidate label strings for scoring
TRUE_VARIANTS = [" True", "True", "\nTrue", " true", "true"]
FALSE_VARIANTS = [" False", "False", "\nFalse", " false", "false"]

# Models
MODELS = {
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
    "google/gemma-7b-it": "causal_chat_or_instruct",
}

# Prompt paraphrases for self-consistency
PARAPHRASE_TEMPLATES = [
    "Statement: {s}\nIs this statement true or false?\nAnswer with only one word: True or False.\nAnswer:",
    "Please determine if the following is true or false:\n{s}\nAnswer (True/False):",
    "Fact-check the statement:\n{s}\nReply with only True or False.\nAnswer:",
    "Analyze this statement for factual accuracy:\n{s}\nAnswer (True/False):",
    "True or False?\nStatement: {s}\nAnswer:",
]

# -----------------------------
# Helpers
# -----------------------------
def subjects_or_empty(subjects: str, use_subjects: bool) -> str:
    return str(subjects) if use_subjects else ""

def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    """
    Applies chat template if supported. Works well for instruct models.
    """
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

def stable_softmax2(loga: float, logb: float):
    """
    Returns (pa, pb) for two log-values using log-sum-exp.
    """
    m = max(loga, logb)
    ea = math.exp(loga - m)
    eb = math.exp(logb - m)
    z = ea + eb
    return ea / z, eb / z

def sequence_logprob(model, tokenizer, rendered_prompt: str, continuation_text: str) -> float:
    """
    Compute log P(continuation_text | rendered_prompt) even if the continuation
    tokenizes into multiple tokens. This avoids the single-token problem.
    """
    prompt_ids = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).input_ids.to(device)
    cont_ids = tokenizer(continuation_text, add_special_tokens=False).input_ids
    if len(cont_ids) == 0:
        return float("-inf")

    cont_ids_t = torch.tensor([cont_ids], device=device, dtype=prompt_ids.dtype)
    full_ids = torch.cat([prompt_ids, cont_ids_t], dim=1)  # [1, T + K]

    with torch.no_grad():
        logits = model(full_ids).logits  # [1, T+K, V]

    prompt_len = prompt_ids.shape[1]
    logp = 0.0
    for j, tok in enumerate(cont_ids):
        pos = prompt_len + j - 1
        if pos < 0:
            continue
        token_logits = logits[0, pos, :]
        token_logprob = torch.log_softmax(token_logits, dim=-1)[tok].item()
        logp += token_logprob
    return logp

def score_true_false(model, tokenizer, rendered_prompt: str):
    """
    Returns:
      pred_bin (1=True, 0=False),
      pred_str,
      confidence in [0,1],
      p_true, p_false (normalized over the two labels)
    We take the best logprob over several label surface forms (variants).
    """
    lp_true = max(sequence_logprob(model, tokenizer, rendered_prompt, v) for v in TRUE_VARIANTS)
    lp_false = max(sequence_logprob(model, tokenizer, rendered_prompt, v) for v in FALSE_VARIANTS)
    p_true, p_false = stable_softmax2(lp_true, lp_false)

    if p_true >= p_false:
        return 1, "True", p_true, p_true, p_false
    else:
        return 0, "False", p_false, p_true, p_false

def vote_self_consistency(p_true_list, p_false_list, weighted=True):
    """
    Weighted vote by probabilities, or unweighted majority vote.
    Confidence is normalized vote share for the winning label.
    """
    v_true = 0.0
    v_false = 0.0

    for pt, pf in zip(p_true_list, p_false_list):
        if weighted:
            if pt >= pf:
                v_true += pt
            else:
                v_false += pf
        else:
            if pt >= pf:
                v_true += 1.0
            else:
                v_false += 1.0

    if v_true + v_false == 0:
        return 1, "True", 0.5

    pred_bin = 1 if v_true >= v_false else 0
    pred_str = "True" if pred_bin == 1 else "False"
    conf = max(v_true, v_false) / (v_true + v_false)
    return pred_bin, pred_str, conf

# -----------------------------
# Prompt families
# -----------------------------
def prompt_base(statement: str, subjects: str, use_subjects: bool) -> str:
    sb = subjects_or_empty(subjects, use_subjects)
    if use_subjects:
        return (
            f"Subjects: {sb}\n"
            f"Statement: {statement}\n"
            f"Is this statement true or false?\n"
            f"Answer with only one word: True or False.\n"
            f"Answer:"
        )
    else:
        return (
            f"Statement: {statement}\n"
            f"Is this statement true or false?\n"
            f"Answer with only one word: True or False.\n"
            f"Answer:"
        )

def prompt_subject(statement: str, subjects: str, use_subjects: bool) -> str:
    sb = subjects_or_empty(subjects, use_subjects)
    if use_subjects:
        return (
            f"Statement: {statement}\n"
            f"Subjects: {sb}\n"
            f"Task: Is the statement factually correct considering these subjects?\n"
            f"Answer only with True or False.\n"
            f"Answer:"
        )
    else:
        return (
            f"Statement: {statement}\n"
            f"Task: Is the statement factually correct?\n"
            f"Answer only with True or False.\n"
            f"Answer:"
        )

# -----------------------------
# Methods
# -----------------------------
def run_baseline(model, tokenizer, model_type, statement, subjects, use_subjects):
    """
    Single prompt, single decision.
    """
    prompt = prompt_base(statement, subjects, use_subjects)
    rendered = maybe_apply_chat_template(tokenizer, prompt, model_type)
    pred_bin, pred_str, conf, p_true, p_false = score_true_false(model, tokenizer, rendered)
    return pred_bin, pred_str, conf, "base"

def run_self_consistency(model, tokenizer, model_type, statement, subjects, use_subjects):
    """
    Self-consistency only: N paraphrases of the BASE instruction.
    """
    base = prompt_base(statement, subjects, use_subjects)
    variants = [base]
    extras = random.sample(PARAPHRASE_TEMPLATES, k=min(N_VARIANTS - 1, len(PARAPHRASE_TEMPLATES)))
    variants += [tpl.format(s=statement) for tpl in extras]
    variants = variants[:N_VARIANTS]

    p_trues, p_falses = [], []
    for p in variants:
        rendered = maybe_apply_chat_template(tokenizer, p, model_type)
        _, _, _, pt, pf = score_true_false(model, tokenizer, rendered)
        p_trues.append(pt)
        p_falses.append(pf)

    pred_bin, pred_str, conf = vote_self_consistency(p_trues, p_falses, weighted=WEIGHTED_VOTE)
    return pred_bin, pred_str, conf, "self_consistency"

def run_reprompting(model, tokenizer, model_type, statement, subjects, use_subjects):
    """
    Reprompting only: staged escalation by confidence threshold.
    No voting; each stage uses one prompt.
    """
    # Stage 1: base
    p = prompt_base(statement, subjects, use_subjects)
    rendered = maybe_apply_chat_template(tokenizer, p, model_type)
    pred_bin, pred_str, conf, _, _ = score_true_false(model, tokenizer, rendered)
    if conf >= CONFIDENCE_THRESHOLD:
        return pred_bin, pred_str, conf, "reprompt_base"

    # Stage 3: subject prompt
    p = prompt_subject(statement, subjects, use_subjects)
    rendered = maybe_apply_chat_template(tokenizer, p, model_type)
    pred_bin, pred_str, conf, _, _ = score_true_false(model, tokenizer, rendered)
    return pred_bin, pred_str, conf, "reprompt_subject"

METHODS = [
    ("baseline", run_baseline),
    ("self_consistency", run_self_consistency),
    ("reprompting", run_reprompting),
]

# -----------------------------
# MAIN
# -----------------------------
def run():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    # Ensure columns
    if STATEMENT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"Missing required columns. Found: {list(df.columns)}")
    if SUBJECTS_COL not in df.columns:
        df[SUBJECTS_COL] = ""

    df[STATEMENT_COL] = df[STATEMENT_COL].astype(str)
    df[SUBJECTS_COL] = df[SUBJECTS_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    pred_rows = []
    summary_rows = []

    for model_name, model_type in MODELS.items():
        print(f"\n=== Loading: {model_name} ({model_type}) ===")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if device.type == "cuda" else None,
        ).to(device)
        model.eval()

        for method_name, method_fn in METHODS:
            for use_subjects in [False, True]:
                mode = "with_subjects" if use_subjects else "no_subjects"
                desc = f"{model_name.split('/')[-1]} | {method_name} | {mode}"
                iterator = tqdm(df.itertuples(index=True), total=len(df), desc=desc, leave=True)

                trues, preds = [], []
                confs, conf_correct, conf_wrong = [], [], []
                stage_counts = {}

                for r in iterator:
                    idx = r.Index
                    statement = getattr(r, STATEMENT_COL)
                    subjects = getattr(r, SUBJECTS_COL)
                    y_true = getattr(r, LABEL_COL)

                    if method_name == "reprompting":
                        y_pred, y_pred_str, conf, stage = method_fn(
                            model, tokenizer, model_type, statement, subjects, use_subjects
                        )
                    else:
                        y_pred, y_pred_str, conf, stage = method_fn(
                            model, tokenizer, model_type, statement, subjects, use_subjects_attach:=None
                        )

                    stage_counts[stage] = stage_counts.get(stage, 0) + 1

                    pred_rows.append({
                        "idx": idx,
                        "model": model_name,
                        "model_short": model_name.split("/")[-1],
                        "method": method_name,
                        "mode": mode,
                        "use_subjects": use_subjects,
                        "y_true": y_true,
                        "y_pred": y_pred,
                        "y_pred_str": y_pred_str,
                        "confidence": float(conf),
                        "stage": stage,
                    })

                    trues.append(y_true)
                    preds.append(y_pred)
                    confs.append(conf)
                    if y_pred == y_true:
                        conf_correct.append(conf)
                    else:
                        conf_wrong.append(conf)

                acc = accuracy_score(trues, preds)
                macro_f1 = f1_score(trues, preds, average="macro")
                cm = confusion_matrix(trues, preds, labels=[0, 1])
                tn, fp, fn, tp = cm.ravel()

                summary_rows.append({
                    "model": model_name,
                    "model_short": model_name.split("/")[-1],
                    "method": method_name,
                    "mode": mode,
                    "accuracy": acc,
                    "macro_f1": macro_f1,
                    "n": len(trues),
                    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
                    "mean_confidence": float(sum(confs) / len(confs)) if confs else None,
                    "mean_conf_correct": float(sum(conf_correct) / len(conf_correct)) if conf_correct else None,
                    "mean_conf_wrong": float(sum(conf_wrong) / len(conf_wrong)) if conf_wrong else None,
                    "stage_counts": dict(stage_counts),
                })

        # cleanup
        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    pred_df = pd.DataFrame(pred_rows)
    pred_df.to_csv(OUT_PRED_PATH, index=False)

    summary_df = pd.DataFrame(summary_rows).sort_values(["accuracy", "macro_f1"], ascending=False)
    summary_df.to_csv(OUT_SUMMARY_PATH, index=False)

    print(f"\nSaved predictions: {OUT_PRED_PATH}")
    print(f"Saved summary: {OUT_SUMMARY_PATH}\n")

    show_cols = [
        "model_short", "method", "mode",
        "accuracy", "macro_f1", "n", "tn", "fp", "fn", "tp",
        "mean_confidence", "mean_conf_correct", "mean_conf_wrong"
    ]
    print(summary_df[show_cols].head(30).to_string(index=False))

if __name__ == "__main__":
    run()



=== Loading: meta-llama/Llama-3.1-8B-Instruct (causal_chat_or_instruct) ===


`torch_dtype` is deprecated! Use `dtype` instead!
E0000 00:00:1767195395.914897  158263 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767195395.919801  158263 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767195395.933594  158263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767195395.933605  158263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767195395.933607  158263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767195395.933610  158263 computation_placer.cc:177] computa

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | baseline | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | baseline | with_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

...

gemma-7b-it | self_consistency | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

gemma-7b-it | self_consistency | with_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

gemma-7b-it | reprompting | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

gemma-7b-it | reprompting | with_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]


Saved predictions: 1_Basics_Masterthesis/Results/reprompt_vs_selfcons_predictions.csv
Saved summary: 1_Basics_Masterthesis/Results/reprompt_vs_selfcons_summary.csv

          model_short           method          mode  accuracy  macro_f1    n  tn  fp  fn  tp  mean_confidence  mean_conf_correct  mean_conf_wrong
          gemma-7b-it      reprompting with_subjects  0.616417  0.583738 1267 213 340 146 568         0.979305           0.982882         0.973558
          gemma-7b-it      reprompting   no_subjects  0.614049  0.591126 1267 239 314 175 539         0.978525           0.980451         0.975460
          gemma-7b-it         baseline   no_subjects  0.612470  0.583900 1267 222 331 160 554         0.960899           0.965920         0.952964
          gemma-7b-it         baseline with_subjects  0.612470  0.583900 1267 222 331 160 554         0.960899           0.965920         0.952964
          gemma-7b-it self_consistency   no_subjects  0.608524  0.583767 1267 231 322 174 540      

### Conclusion

Across both models, subjects have only a minor impact on performance; differences between with_subjects and no_subjects are small and not consistent in one direction. The main effect comes from the method: Gemma-7B benefits most from reprompting, improving over both baseline and self-consistency and reaching the best overall accuracy (~0.616). Self-consistency provides a small but stable gain over the baseline for both models (especially for Llama), suggesting that aggregating multiple prompt variants reduces some variance and improves robustness.

For Llama-3.1, self-consistency improves accuracy over baseline (~0.538–0.541 vs. ~0.529), but reprompting does not help and can even hurt (down to ~0.502 in the no-subjects condition). This indicates that Llama’s decisions are more sensitive to prompt changes, and reprompting may push it toward a less reliable decision boundary for this dataset and setup.

Finally, the confidence scores are high for reprompting, but the Llama results show that higher confidence does not necessarily imply higher correctness—so confidence should be interpreted carefully and ideally calibrated/validated (e.g., via confidence vs. accuracy curves or ECE later).

## iterative refinement

In [ ]:
import os
import re
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
N_ROWS = None

LABEL_COL = "label"
STATEMENT_COL = "statement"
SUBJECTS_COL = "subjects"

OUT_PRED_PATH = "1_Basics_Masterthesis/Results/iter_refinement_predictions.csv"
OUT_SUMMARY_PATH = "1_Basics_Masterthesis/Results/iter_refinement_summary.csv"

# iterative refinement settings
MAX_REFINEMENT_STEPS = 3
CONFIDENCE_THRESHOLD = 0.75

# Models
MODELS = {
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
    "google/gemma-7b-it": "causal_chat_or_instruct",
}

# -----------------------------
# PROMPTS
# -----------------------------
def subjects_or_empty(subjects: str, use_subjects: bool) -> str:
    return str(subjects) if use_subjects else ""

def initial_prompt(statement: str, subjects: str, use_subjects: bool) -> str:
    sb = subjects_or_empty(subjects, use_subjects)
    if use_subjects:
        return (
            f"Subjects: {sb}\n"
            f"Statement: {statement}\n"
            f"Is this statement true or false?\n"
            f"Answer with only one word: True or False.\n"
            f"Answer:"
        )
    else:
        return (
            f"Statement: {statement}\n"
            f"Is this statement true or false?\n"
            f"Answer with only one word: True or False.\n"
            f"Answer:"
        )

def refinement_prompt(statement: str, subjects: str, use_subjects: bool, previous_answer_text: str) -> str:
    """
    Uses the previous model output (free-text) as the thing to critique.
    Still ends with binary-only instruction.
    """
    sb = subjects_or_empty(subjects, use_subjects)
    prefix = f"Subjects: {sb}\n" if use_subjects else ""
    return (
        prefix +
        f"Statement: {statement}\n"
        f"You previously answered:\n{previous_answer_text}\n\n"
        f"Now reflect critically: Is your answer still correct? If not, revise it.\n"
        f"Answer again with only one word: True or False.\n"
        f"Answer:"
    )

# -----------------------------
# TOKEN + SCORING HELPERS (Setup-B-like)
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

def get_single_token_id(tokenizer, text_variants):
    for txt in text_variants:
        ids = tokenizer.encode(txt, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return None

def true_false_token_ids(tokenizer):
    true_id = get_single_token_id(tokenizer, [" True", " true", "True", "true"])
    false_id = get_single_token_id(tokenizer, [" False", " false", "False", "false"])
    if true_id is None or false_id is None:
        raise ValueError("Could not find single-token IDs for True/False with this tokenizer.")
    return true_id, false_id

def next_token_probs(model, tokenizer, rendered_prompt: str, true_id: int, false_id: int):
    """
    Computes p(True) and p(False) for the *next token* after the prompt.
    Confidence = max(p_true, p_false).
    """
    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits  # [1, T, V]
    next_logits = logits[0, -1, :]   # [V]
    probs = F.softmax(next_logits, dim=-1)
    p_true = probs[true_id].item()
    p_false = probs[false_id].item()
    return p_true, p_false

def decide_from_probs(p_true: float, p_false: float):
    pred_bin = 1 if p_true >= p_false else 0
    pred_str = "True" if pred_bin == 1 else "False"
    conf = max(p_true, p_false)
    return pred_bin, pred_str, conf

def generate_free_text(model, tokenizer, rendered_prompt: str, max_new_tokens: int = 80):
    """
    We generate a short explanation text ONLY to feed into the refinement step.
    Final decision is still taken via next-token scoring (Setup-B-like).
    """
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.eos_token_id,
    )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    completion = decoded[len(rendered_prompt):].strip()
    if not completion:
        completion = decoded.strip()
    return completion

# -----------------------------
# ITERATIVE SELF REFINEMENT
# -----------------------------
def iterative_refinement_one(
    statement: str,
    subjects: str,
    use_subjects: bool,
    tokenizer,
    model,
    model_type: str,
    true_id: int,
    false_id: int,
):
    """
    Returns final (pred_bin, pred_str, conf, steps_used) plus per-step debug rows.
    """
    debug_steps = []

    # Step 0: initial
    p0 = initial_prompt(statement, subjects, use_subjects)
    r0 = maybe_apply_chat_template(tokenizer, p0, model_type)

    # Score decision (Setup-B-like)
    p_true, p_false = next_token_probs(model, tokenizer, r0, true_id, false_id)
    pred_bin, pred_str, conf = decide_from_probs(p_true, p_false)

    # Generate explanation text to refine
    completion_text = generate_free_text(model, tokenizer, r0, max_new_tokens=80)

    debug_steps.append({
        "step": 0,
        "prompt_kind": "initial",
        "p_true": p_true,
        "p_false": p_false,
        "pred_bin": pred_bin,
        "pred_str": pred_str,
        "confidence": conf,
        "completion": completion_text,
    })

    if conf >= CONFIDENCE_THRESHOLD:
        return pred_bin, pred_str, conf, 0, debug_steps

    prev_answer_text = completion_text
    for step in range(1, MAX_REFINEMENT_STEPS + 1):
        pr = refinement_prompt(statement, subjects, use_subjects, prev_answer_text)
        rr = maybe_apply_chat_template(tokenizer, pr, model_type)

        p_true, p_false = next_token_probs(model, tokenizer, rr, true_id, false_id)
        pred_bin, pred_str, conf = decide_from_probs(p_true, p_false)

        completion_text = generate_free_text(model, tokenizer, rr, max_new_tokens=80)

        debug_steps.append({
            "step": step,
            "prompt_kind": "refinement",
            "p_true": p_true,
            "p_false": p_false,
            "pred_bin": pred_bin,
            "pred_str": pred_str,
            "confidence": conf,
            "completion": completion_text,
        })

        if conf >= CONFIDENCE_THRESHOLD:
            return pred_bin, pred_str, conf, step, debug_steps

        prev_answer_text = completion_text

    # if never reached threshold: return last
    return pred_bin, pred_str, conf, MAX_REFINEMENT_STEPS, debug_steps

# -----------------------------
# MAIN
# -----------------------------
def run():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    # allow running even if subjects missing
    if SUBJECTS_COL not in df.columns:
        df[SUBJECTS_COL] = ""

    for col in [STATEMENT_COL, SUBJECTS_COL, LABEL_COL]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}'. Found: {list(df.columns)}")

    df[STATEMENT_COL] = df[STATEMENT_COL].astype(str)
    df[SUBJECTS_COL] = df[SUBJECTS_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    pred_rows = []
    summary_rows = []

    for model_name, model_type in MODELS.items():
        print(f"\n=== Loading: {model_name} ({model_type}) ===")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if device.type == "cuda" else None,
        ).to(device)
        model.eval()

        true_id, false_id = true_false_token_ids(tokenizer)

        for use_subjects in [False, True]:
            mode = "with_subjects" if use_subjects else "no_subjects"
            desc = f"{model_name.split('/')[-1]} | iterative_refinement | {mode}"
            iterator = tqdm(df.itertuples(index=True), total=len(df), desc=desc, leave=True)

            trues, preds = [], []
            confs = []
            conf_correct = []
            conf_wrong = []
            steps_used_list = []

            for r in iterator:
                idx = r.Index
                statement = getattr(r, STATEMENT_COL)
                subjects = getattr(r, SUBJECTS_COL)
                y_true = getattr(r, LABEL_COL)

                y_pred, y_pred_str, conf, steps_used, step_debug = iterative_refinement_one(
                    statement, subjects, use_subjects,
                    tokenizer, model, model_type,
                    true_id, false_id
                )

                # store final row
                pred_rows.append({
                    "idx": idx,
                    "model": model_name,
                    "model_short": model_name.split("/")[-1],
                    "mode": mode,
                    "use_subjects": use_subjects,
                    "y_true": y_true,
                    "y_pred": y_pred,
                    "y_pred_str": y_pred_str,
                    "confidence": conf,
                    "steps_used": steps_used,
                })

                trues.append(y_true)
                preds.append(y_pred)
                confs.append(conf)
                steps_used_list.append(steps_used)

                if y_pred == y_true:
                    conf_correct.append(conf)
                else:
                    conf_wrong.append(conf)

            acc = accuracy_score(trues, preds)
            macro_f1 = f1_score(trues, preds, average="macro")
            cm = confusion_matrix(trues, preds, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()

            summary_rows.append({
                "model": model_name,
                "model_short": model_name.split("/")[-1],
                "mode": mode,
                "accuracy": acc,
                "macro_f1": macro_f1,
                "n": len(trues),
                "tn": tn, "fp": fp, "fn": fn, "tp": tp,
                "mean_confidence": float(sum(confs) / len(confs)) if confs else None,
                "mean_conf_correct": float(sum(conf_correct) / len(conf_correct)) if conf_correct else None,
                "mean_conf_wrong": float(sum(conf_wrong) / len(conf_wrong)) if conf_wrong else None,
                "mean_steps_used": float(sum(steps_used_list) / len(steps_used_list)) if steps_used_list else None,
            })

        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    pred_df = pd.DataFrame(pred_rows)
    pred_df.to_csv(OUT_PRED_PATH, index=False)

    summary_df = pd.DataFrame(summary_rows).sort_values(["accuracy", "macro_f1"], ascending=False)
    summary_df.to_csv(OUT_SUMMARY_PATH, index=False)

    print(f"\nSaved predictions: {OUT_PRED_PATH}")
    print(f"Saved summary: {OUT_SUMMARY_PATH}")
    print("\nSummary:")
    print(summary_df.to_string(index=False))

if __name__ == "__main__":
    run()



=== Loading: meta-llama/Llama-3.1-8B-Instruct (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | iterative_refinement | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [10]:
df = pd.read_csv("1_Basics_Masterthesis/Results/iter_refinement_summary.csv")
df

,model,model_short,mode,accuracy,macro_f1,n,tn,fp,fn,tp,mean_confidence,mean_conf_correct,mean_conf_wrong,mean_steps_used
0,google/gemma-7b-it,gemma-7b-it,no_subjects,0.564325,0.364062,1267,2,551,1,713,2.850861e-07,2.728476e-07,3.009387e-07,3.0
1,google/gemma-7b-it,gemma-7b-it,with_subjects,0.562747,0.360101,1267,0,553,1,713,8.355000e-08,7.991871e-08,8.822348e-08,3.0
2,meta-llama/Llama-3.1-8B-Instruct,Llama-3.1-8B-Instruct,with_subjects,0.519337,0.499210,1267,456,97,512,202,5.365520e-04,5.281353e-04,5.456459e-04,3.0
3,meta-llama/Llama-3.1-8B-Instruct,Llama-3.1-8B-Instruct,no_subjects,0.503552,0.471504,1267,475,78,551,163,5.368355e-04,5.406603e-04,5.329560e-04,3.0


### Conclusion

Short conclusion (Iterative Self-Refinement)

Iterative self-refinement did not improve performance for either model and in some cases substantially degraded it. Gemma collapses into an almost single-class behavior (very high FN/TP imbalance), yielding Acc ≈ 0.563–0.564 with Macro-F1 ≈ 0.36, far below its baseline/prompt-engineering results. Llama also declines compared to its baseline/self-consistency, reaching only Acc ≈ 0.504–0.519 with moderate Macro-F1.

A key signal is that mean_steps_used = 3.0 everywhere (always hitting the maximum), meaning the refinement loop rarely reached the confidence threshold early and therefore kept “refining” without actually becoming more certain or more correct. In practice, this indicates iterative refinement (as implemented here) tends to amplify bias or drift rather than correct mistakes for this task.

Subjects again show negligible impact compared to the dominant effect of the method itself.

## Chain of Thought

In [ ]:
import os
import re
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
N_ROWS = None

LABEL_COL = "label"
STATEMENT_COL = "statement"
SUBJECTS_COL = "subjects"

OUT_PRED_PATH = "1_Basics_Masterthesis/Results/cot_predictions.csv"
OUT_SUMMARY_PATH = "1_Basics_Masterthesis/Results/cot_summary.csv"

# Generation settings (reasoning + final answer)
MAX_NEW_TOKENS = 120

# Models
MODELS = {
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
    "google/gemma-7b-it": "causal_chat_or_instruct",
}

# If no True/False appears , force fallback like Setup B
FALLBACK_LABEL_STR = "True"
FALLBACK_LABEL_BIN = 1 if FALLBACK_LABEL_STR == "True" else 0
TAKE_OCCURRENCE = "last"

# -----------------------------
# PROMPTS
# -----------------------------
def subjects_or_empty(subjects: str, use_subjects: bool) -> str:
    return str(subjects) if use_subjects else ""

def cot_prompt(statement: str, subjects: str, use_subjects: bool) -> str:
    sb = subjects_or_empty(subjects, use_subjects)
    prefix = f"Subjects: {sb}\n" if use_subjects else ""
    return (
        prefix
        + f"Statement: {statement}\n"
          f"Task: Determine whether the statement is factually correct.\n"
          f"Provide a short reasoning (1–3 sentences) based on publicly verifiable facts.\n"
          f"Then output exactly one final line in the format:\n"
          f"Final answer: True\n"
          f"or\n"
          f"Final answer: False\n"
          f"Reasoning:"
    )

# -----------------------------
# CHAT TEMPLATE
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

# -----------------------------
# FORCED PARSE (Setup-B-like)
# -----------------------------
def parse_true_false_forced(text: str):
    if text is None:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN
    s = str(text).strip().lower()
    matches = re.findall(r"\b(true|false)\b", s)
    if not matches:
        return FALLBACK_LABEL_STR, FALLBACK_LABEL_BIN
    choice = matches[0] if TAKE_OCCURRENCE == "first" else matches[-1]
    return ("True", 1) if choice == "true" else ("False", 0)

# -----------------------------
# TOKEN HELPERS FOR CONFIDENCE
# -----------------------------
def get_single_token_id(tokenizer, text_variants):
    for txt in text_variants:
        ids = tokenizer.encode(txt, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return None

def true_false_token_ids(tokenizer):
    true_id = get_single_token_id(tokenizer, [" True", " true", "True", "true"])
    false_id = get_single_token_id(tokenizer, [" False", " false", "False", "false"])
    if true_id is None or false_id is None:
        raise ValueError("Could not find single-token IDs for True/False for this tokenizer.")
    return true_id, false_id

def next_token_probs_from_text(model, tokenizer, prefix_text: str, true_id: int, false_id: int):
    """
    Scores p(True) and p(False) as the NEXT token after prefix_text.
    """
    inputs = tokenizer(prefix_text, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits
    next_logits = logits[0, -1, :]
    probs = F.softmax(next_logits, dim=-1)
    return probs[true_id].item(), probs[false_id].item()

def confidence_after_final_answer_marker(model, tokenizer, full_generated_text: str, true_id: int, false_id: int):
    """
    Computes confidence at the position right after 'Final answer:' (conditioned on reasoning).
    If marker not found, fall back to scoring after the original text.
    """
    marker = "final answer:"
    lower = full_generated_text.lower()
    pos = lower.rfind(marker)
    if pos == -1:
        # fallback: score on whole text end
        pt, pf = next_token_probs_from_text(model, tokenizer, full_generated_text, true_id, false_id)
        return max(pt, pf), pt, pf

    prefix = full_generated_text[: pos + len(marker)]
    pt, pf = next_token_probs_from_text(model, tokenizer, prefix, true_id, false_id)
    return max(pt, pf), pt, pf

# -----------------------------
# GENERATION
# -----------------------------
def generate_text(model, tokenizer, rendered_prompt: str, max_new_tokens: int):
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    input_len = inputs["input_ids"].shape[1]

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.eos_token_id,
    )

    # Decode FULL text, and completion separately
    full_text = tokenizer.decode(out[0], skip_special_tokens=True)
    completion_ids = out[0][input_len:]
    completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True)

    return full_text, completion_text

# -----------------------------
# MAIN
# -----------------------------
def run():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    if SUBJECTS_COL not in df.columns:
        df[SUBJECTS_COL] = ""

    for col in [STATEMENT_COL, SUBJECTS_COL, LABEL_COL]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}'. Found: {list(df.columns)}")

    df[STATEMENT_COL] = df[STATEMENT_COL].astype(str)
    df[SUBJECTS_COL] = df[SUBJECTS_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    pred_rows = []
    summary_rows = []

    for model_name, model_type in MODELS.items():
        print(f"\n=== Loading: {model_name} ({model_type}) ===")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if device.type == "cuda" else None,
        ).to(device)
        model.eval()

        true_id, false_id = true_false_token_ids(tokenizer)

        for use_subjects in [False, True]:
            mode = "with_subjects" if use_subjects else "no_subjects"
            desc = f"{model_name.split('/')[-1]} | CoT | {mode}"
            iterator = tqdm(df.itertuples(index=True), total=len(df), desc=desc, leave=True)

            trues, preds = [], []
            confs, conf_correct, conf_wrong = [], [], []

            for r in iterator:
                idx = r.Index
                statement = getattr(r, STATEMENT_COL)
                subjects = getattr(r, SUBJECTS_COL)
                y_true = getattr(r, LABEL_COL)

                prompt = cot_prompt(statement, subjects, use_subjects)
                rendered = maybe_apply_chat_template(tokenizer, prompt, model_type)

                full_text, completion = generate_text(model, tokenizer, rendered, MAX_NEW_TOKENS)

                # Prediction from generated output (Setup-B-like forced parse)
                pred_str, pred_bin = parse_true_false_forced(full_text)

                # Confidence *at* the decision point after "Final answer:"
                conf, p_true, p_false = confidence_after_final_answer_marker(
                    model, tokenizer, full_text, true_id, false_id
                )

                pred_rows.append({
                    "idx": idx,
                    "model": model_name,
                    "model_short": model_name.split("/")[-1],
                    "mode": mode,
                    "use_subjects": use_subjects,
                    "y_true": y_true,
                    "y_pred": pred_bin,
                    "y_pred_str": pred_str,
                    "confidence": conf,
                    "p_true": p_true,
                    "p_false": p_false,
                    "raw_completion": completion,   # reasoning+answer (completion only)
                })

                trues.append(y_true)
                preds.append(pred_bin)
                confs.append(conf)
                if pred_bin == y_true:
                    conf_correct.append(conf)
                else:
                    conf_wrong.append(conf)

            acc = accuracy_score(trues, preds)
            macro_f1 = f1_score(trues, preds, average="macro")
            cm = confusion_matrix(trues, preds, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()

            summary_rows.append({
                "model": model_name,
                "model_short": model_name.split("/")[-1],
                "mode": mode,
                "accuracy": acc,
                "macro_f1": macro_f1,
                "n": len(trues),
                "tn": tn, "fp": fp, "fn": fn, "tp": tp,
                "mean_confidence": float(sum(confs) / len(confs)) if confs else None,
                "mean_conf_correct": float(sum(conf_correct) / len(conf_correct)) if conf_correct else None,
                "mean_conf_wrong": float(sum(conf_wrong) / len(conf_wrong)) if conf_wrong else None,
            })

        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    pred_df = pd.DataFrame(pred_rows)
    pred_df.to_csv(OUT_PRED_PATH, index=False)

    summary_df = pd.DataFrame(summary_rows).sort_values(["accuracy", "macro_f1"], ascending=False)
    summary_df.to_csv(OUT_SUMMARY_PATH, index=False)

    print(f"\nSaved predictions: {OUT_PRED_PATH}")
    print(f"Saved summary: {OUT_SUMMARY_PATH}")
    print("\nSummary:")
    print(summary_df.to_string(index=False))

if __name__ == "__main__":
    run()


In [11]:
df = pd.read_csv("1_Basics_Masterthesis/Results/cot_summary.csv")
df

,model,model_short,mode,accuracy,macro_f1,n,tn,fp,fn,tp,mean_confidence,mean_conf_correct,mean_conf_wrong
0,google/gemma-7b-it,gemma-7b-it,no_subjects,0.603788,0.601153,1267,331,222,280,434,0.987163,0.989097,0.984215
1,google/gemma-7b-it,gemma-7b-it,with_subjects,0.603788,0.596611,1267,298,255,247,467,0.990783,0.993050,0.987328
2,meta-llama/Llama-3.1-8B-Instruct,Llama-3.1-8B-Instruct,with_subjects,0.505919,0.475527,1267,473,80,546,168,0.943857,0.950737,0.936811
3,meta-llama/Llama-3.1-8B-Instruct,Llama-3.1-8B-Instruct,no_subjects,0.491713,0.449460,1267,487,66,578,136,0.933047,0.948999,0.917614


### Conclusion

Chain-of-thought prompting produces strong confidence values, but does not translate into better accuracy, especially for Llama. Gemma achieves Acc ≈ 0.604 with Macro-F1 ≈ 0.597–0.601, which is slightly below its best prompt-engineering / reprompting results, but still competitive. In contrast, Llama performs poorly (Acc ≈ 0.492–0.506), worse than even its baseline and self-consistency runs, suggesting that CoT-style prompting increases verbosity but not reliability for this binary classification setting.

Subjects again have no consistent benefit: Gemma is marginally better without subjects; Llama is marginally better with subjects, but differences are small.

Overall takeaway: CoT is not a robust improvement lever here—Gemma remains fairly stable, but Llama degrades—so for the next experiments, reprompting/self-consistency are the more promising directions than CoT.

# Tone classification

Analyzes the linguistic tone of statements (neutral, subjective, ironic, emotional) to understand how statement characteristics correlate with model predictions and factuality.

In [ ]:
import os
import re
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
N_ROWS = None

STATEMENT_COL = "statement"
SUBJECTS_COL = "subjects"

OUT_TONE_PATH = "1_Basics_Masterthesis/Results/statements_tone.csv"
OUT_NEUTRAL_PATH = "1_Basics_Masterthesis/Results/statements_neutralized.csv"

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_TYPE = "causal_chat_or_instruct"

TONE_LABELS = ["neutral", "subjective", "ironic", "emotional"]

# -----------------------------
# Chat template
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

# -----------------------------
# Prompts
# -----------------------------
def tone_classification_prompt(statement: str, subjects: str = None) -> str:
    prefix = f"Subjects: {subjects}\n" if subjects is not None and str(subjects).strip() != "" else ""
    return (
        prefix +
        "Classify the tone of the following statement.\n"
        "Choose exactly one label from: neutral, subjective, ironic, emotional.\n\n"
        f"Statement: {statement}\n"
        "Tone:"
    )

def neutral_rewrite_prompt(statement: str, subjects: str = None) -> str:
    prefix = f"Subjects: {subjects}\n" if subjects is not None and str(subjects).strip() != "" else ""
    return (
        prefix +
        "Rewrite the following statement in a neutral, objective tone.\n"
        "Remove irony, sarcasm, exaggeration, and emotionally loaded phrasing.\n"
        "Do NOT add new facts. Keep the meaning as close as possible.\n\n"
        f"Original: {statement}\n"
        "Neutral:"
    )

# -----------------------------
# Parsing
# -----------------------------
def parse_tone_label(text: str):
    if text is None:
        return "neutral"
    s = str(text).strip().lower()
    for lab in TONE_LABELS:
        if re.search(rf"\b{re.escape(lab)}\b", s):
            return lab
    # fallback
    return "neutral"

def extract_after_marker(full_text: str, marker: str):
    if full_text is None:
        return ""
    if marker in full_text:
        return full_text.split(marker, 1)[-1].strip()
    return full_text.strip()

# -----------------------------
# Generation
# -----------------------------
def generate_text(model, tokenizer, rendered_prompt: str, max_new_tokens: int):
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# -----------------------------
# MAIN
# -----------------------------
def run():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    if STATEMENT_COL not in df.columns:
        raise ValueError(f"Missing column '{STATEMENT_COL}'. Found: {list(df.columns)}")
    if SUBJECTS_COL not in df.columns:
        df[SUBJECTS_COL] = ""

    df[STATEMENT_COL] = df[STATEMENT_COL].astype(str)
    df[SUBJECTS_COL] = df[SUBJECTS_COL].astype(str)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=torch.float16 if device.type == "cuda" else None,
    ).to(device)
    model.eval()

    tone_preds = []
    neutral_rewrites = []

    iterator = tqdm(df.itertuples(index=False), total=len(df), desc="tone+neutralize", leave=True)
    for r in iterator:
        statement = getattr(r, STATEMENT_COL)
        subjects = getattr(r, SUBJECTS_COL)

        # Tone classification
        p_tone = tone_classification_prompt(statement, subjects)
        r_tone = maybe_apply_chat_template(tokenizer, p_tone, MODEL_TYPE)
        out_tone = generate_text(model, tokenizer, r_tone, max_new_tokens=8)
        tone_label = parse_tone_label(out_tone)
        tone_preds.append(tone_label)

        # Neutral rewrite
        p_neu = neutral_rewrite_prompt(statement, subjects)
        r_neu = maybe_apply_chat_template(tokenizer, p_neu, MODEL_TYPE)
        out_neu = generate_text(model, tokenizer, r_neu, max_new_tokens=80)
        neutral_text = extract_after_marker(out_neu, "Neutral:")
        neutral_rewrites.append(neutral_text)

    # Save tone labels
    df_tone = df.copy()
    df_tone["tone"] = tone_preds
    df_tone.to_csv(OUT_TONE_PATH, index=False)

    # Save neutral rewrites
    df_neu = df.copy()
    df_neu["neutral_statement"] = neutral_rewrites
    df_neu.to_csv(OUT_NEUTRAL_PATH, index=False)

    print(f"\nSaved tone labels: {OUT_TONE_PATH}")
    print(f"Saved neutral rewrites: {OUT_NEUTRAL_PATH}")

if __name__ == "__main__":
    run()


In [ ]:
import os
import re
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
N_ROWS = None

LABEL_COL = "label"
STATEMENT_COL = "statement"
SUBJECTS_COL = "subjects"

# Produced earlier by your tone pipeline
TONE_CSV = "1_Basics_Masterthesis/Results/statements_tone.csv"           
NEUTRAL_CSV = "1_Basics_Masterthesis/Results/statements_neutralized.csv"
TONE_COL = "tone"
NEUTRAL_COL = "neutral_statement"

OUT_PRED_PATH = "1_Basics_Masterthesis/Results/tone_effect_predictions.csv"
OUT_SUMMARY_PATH = "1_Basics_Masterthesis/Results/tone_effect_summary.csv"

MODELS = {
    "google/gemma-7b-it": "causal_chat_or_instruct",
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
}

# Conditions to evaluate
CONDITIONS = ["orig", "neutral", "orig_plus_tone", "neutral_plus_tone"]

# -----------------------------
# Chat template
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

# -----------------------------
# Setup-B scoring helpers (True/False next-token probs)
# -----------------------------
def get_single_token_id(tokenizer, text_variants):
    for txt in text_variants:
        ids = tokenizer.encode(txt, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return None

def true_false_token_ids(tokenizer):
    true_id = get_single_token_id(tokenizer, [" True", " true", "True", "true"])
    false_id = get_single_token_id(tokenizer, [" False", " false", "False", "false"])
    if true_id is None or false_id is None:
        raise ValueError("Could not find single-token IDs for True/False with this tokenizer.")
    return true_id, false_id

def next_token_probs(model, tokenizer, rendered_prompt: str, true_id: int, false_id: int):
    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits  # [1, T, V]
    next_logits = logits[0, -1, :].float()
    probs = torch.softmax(next_logits, dim=-1)
    p_true = probs[true_id].item()
    p_false = probs[false_id].item()
    return p_true, p_false

def decide_from_probs(p_true: float, p_false: float, eps: float = 1e-12):
    pred_bin = 1 if p_true >= p_false else 0
    pred_str = "True" if pred_bin == 1 else "False"

    conf_max = max(p_true, p_false)
    denom = p_true + p_false
    conf_norm = conf_max / (denom + eps)

    return pred_bin, pred_str, conf_max, conf_norm


# -----------------------------
# Prompt building (P0, P4) with optional subjects + optional tone line
# -----------------------------
def subjects_or_empty(subjects: str, use_subjects: bool) -> str:
    return str(subjects) if use_subjects else ""

def tone_line(tone: str, include_tone: bool) -> str:
    if not include_tone:
        return ""
    if tone is None:
        return ""
    t = str(tone).strip().lower()
    if t == "" or t == "nan":
        return ""
    return f"Tone: {t}\n"

def p0_basic(statement: str, subjects: str, tone: str, use_subjects: bool, include_tone: bool) -> str:
    sb = subjects_or_empty(subjects, use_subjects)
    prefix = f"Subjects: {sb}\n" if use_subjects else ""
    return (
        prefix +
        tone_line(tone, include_tone) +
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )

def p4_subjects_as_context_clues(statement: str, subjects: str, tone: str, use_subjects: bool, include_tone: bool) -> str:
    sb = subjects_or_empty(subjects, use_subjects)
    if use_subjects:
        subj_sentence = f"You may use the following subjects as context clues: {sb}.\n"
    else:
        subj_sentence = ""

    return (
        "You are an AI assistant specialized in fact-checking factual statements.\n"
        "Your task is to determine whether the following statement is factually correct.\n"
        + subj_sentence +
        tone_line(tone, include_tone) +
        "Respond strictly with one word: True or False.\n\n"
        "Example:\n"
        "Statement: The sky is green.\n"
        "Answer: False\n\n"
        f"Statement: {statement}\n"
        "Answer:"
    )

PROMPTS = [
    ("P0_basic", p0_basic),
    ("P4_subjects_as_context_clues", p4_subjects_as_context_clues),
]

# -----------------------------
# Data loading / merge
# -----------------------------
def load_and_merge():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    if SUBJECTS_COL not in df.columns:
        df[SUBJECTS_COL] = ""

    for col in [STATEMENT_COL, SUBJECTS_COL, LABEL_COL]:
        if col not in df.columns:
            raise ValueError(f"Missing column '{col}'. Found: {list(df.columns)}")

    df[STATEMENT_COL] = df[STATEMENT_COL].astype(str)
    df[SUBJECTS_COL] = df[SUBJECTS_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)

    # tone file
    tone_df = pd.read_csv(TONE_CSV)
    if N_ROWS is not None:
        tone_df = tone_df.iloc[:N_ROWS].copy()
    if TONE_COL not in tone_df.columns:
        raise ValueError(f"'{TONE_COL}' missing in {TONE_CSV}. Columns: {list(tone_df.columns)}")
    tone_df[TONE_COL] = tone_df[TONE_COL].astype(str)

    # neutral file
    neu_df = pd.read_csv(NEUTRAL_CSV)
    if N_ROWS is not None:
        neu_df = neu_df.iloc[:N_ROWS].copy()
    if NEUTRAL_COL not in neu_df.columns:
        raise ValueError(f"'{NEUTRAL_COL}' missing in {NEUTRAL_CSV}. Columns: {list(neu_df.columns)}")
    neu_df[NEUTRAL_COL] = neu_df[NEUTRAL_COL].astype(str)

    # merge by row order / index 
    df = df.reset_index(drop=True)
    tone_df = tone_df.reset_index(drop=True)
    neu_df = neu_df.reset_index(drop=True)

    df["tone"] = tone_df[TONE_COL]
    df["neutral_statement"] = neu_df[NEUTRAL_COL]
    return df

# -----------------------------
# MAIN
# -----------------------------
def run():
    df = load_and_merge()

    pred_rows = []
    summary_rows = []

    for model_name, model_type in MODELS.items():
        print(f"\n=== Loading: {model_name} ({model_type}) ===")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if device.type == "cuda" else None,
        ).to(device)
        model.eval()

        true_id, false_id = true_false_token_ids(tokenizer)

        for prompt_name, prompt_fn in PROMPTS:
            for use_subjects in [False, True]:
                mode = "with_subjects" if use_subjects else "no_subjects"

                for condition in CONDITIONS:
                    # map condition -> statement source + include_tone
                    use_neutral = condition.startswith("neutral")
                    include_tone = condition.endswith("plus_tone")

                    desc = f"{model_name.split('/')[-1]} | {prompt_name} | {mode} | {condition}"
                    iterator = tqdm(df.itertuples(index=True), total=len(df), desc=desc, leave=False)

                    trues, preds = [], []
                    conf_max_list, conf_norm_list = [], []

                    for r in iterator:
                        idx = r.Index
                        y_true = getattr(r, LABEL_COL)
                        subjects = getattr(r, SUBJECTS_COL)
                        tone = getattr(r, "tone")

                        statement = getattr(r, "neutral_statement") if use_neutral else getattr(r, STATEMENT_COL)

                        prompt = prompt_fn(statement, subjects, tone, use_subjects, include_tone)
                        rendered = maybe_apply_chat_template(tokenizer, prompt, model_type)

                        p_true, p_false = next_token_probs(model, tokenizer, rendered, true_id, false_id)
                        y_pred, y_pred_str, conf_max, conf_norm = decide_from_probs(p_true, p_false)

                        pred_rows.append({
                            "idx": idx,
                            "model": model_name,
                            "model_short": model_name.split("/")[-1],
                            "prompt": prompt_name,
                            "mode": mode,
                            "condition": condition,
                            "y_true": y_true,
                            "y_pred": y_pred,
                            "y_pred_str": y_pred_str,
                            "p_true": p_true,
                            "p_false": p_false,
                            "conf_max": conf_max,
                            "conf_norm": conf_norm,
                        })

                        trues.append(y_true)
                        preds.append(y_pred)
                        conf_max_list.append(conf_max)
                        conf_norm_list.append(conf_norm)

                    acc = accuracy_score(trues, preds)
                    macro_f1 = f1_score(trues, preds, average="macro")
                    tn, fp, fn, tp = confusion_matrix(trues, preds, labels=[0, 1]).ravel()

                    summary_rows.append({
                        "model": model_name,
                        "model_short": model_name.split("/")[-1],
                        "prompt": prompt_name,
                        "mode": mode,
                        "condition": condition,
                        "accuracy": acc,
                        "macro_f1": macro_f1,
                        "n": len(trues),
                        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
                        "mean_conf_max": float(sum(conf_max_list) / len(conf_max_list)),
                        "mean_conf_norm": float(sum(conf_norm_list) / len(conf_norm_list)),
                    })

        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    pred_df = pd.DataFrame(pred_rows)
    pred_df.to_csv(OUT_PRED_PATH, index=False)

    summary_df = pd.DataFrame(summary_rows).sort_values(["accuracy", "macro_f1"], ascending=False)
    summary_df.to_csv(OUT_SUMMARY_PATH, index=False)

    print(f"\nSaved predictions: {OUT_PRED_PATH}")
    print(f"Saved summary: {OUT_SUMMARY_PATH}")
    print("\nTop 20 summary rows:")
    print(summary_df.head(20).to_string(index=False))

if __name__ == "__main__":
    run()



=== Loading: google/gemma-7b-it (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

gemma-7b-it | P0_basic | no_subjects | orig:   0%|          | 0/1267 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


gemma-7b-it | P0_basic | no_subjects | neutral:   0%|          | 0/1267 [00:00<?, ?it/s]

...

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | with_subjects | orig:   0%|          | 0/1267 [00:00<?,…

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | with_subjects | neutral:   0%|          | 0/1267 [00:00…

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | with_subjects | orig_plus_tone:   0%|          | 0/1267…

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | with_subjects | neutral_plus_tone:   0%|          | 0/1…


Saved predictions: 1_Basics_Masterthesis/Results/tone_effect_predictions.csv
Saved summary: 1_Basics_Masterthesis/Results/tone_effect_summary.csv

Top 20 summary rows:
                           model           model_short                       prompt          mode         condition  accuracy  macro_f1    n  tn  fp  fn  tp  mean_conf_max  mean_conf_norm
              google/gemma-7b-it           gemma-7b-it                     P0_basic with_subjects    orig_plus_tone  0.616417  0.573716 1267 190 363 123 591       0.000021        0.957805
              google/gemma-7b-it           gemma-7b-it P4_subjects_as_context_clues with_subjects    orig_plus_tone  0.614838  0.597495 1267 258 295 193 521       0.000028        0.961353
              google/gemma-7b-it           gemma-7b-it P4_subjects_as_context_clues   no_subjects    orig_plus_tone  0.613260  0.612175 1267 355 198 292 422       0.000018        0.952112
              google/gemma-7b-it           gemma-7b-it                     P0_b

## Conclusion

In the tone-effect experiments, the best results came from using the original statements, not the neutralized rewrites. For Gemma-7B-IT, the top configuration was P0_basic (with subjects) + orig_plus_tone (accuracy ≈ 0.616), closely followed by P4 + orig_plus_tone (accuracy ≈ 0.613) and the original (orig) variants around 0.61. In contrast, neutralized statements consistently reduced performance for Gemma to roughly 0.565–0.578, even when the tone label was provided.

For Llama-3.1-8B-Instruct, overall performance remained lower (best ≈ 0.523 with P0 + neutral_plus_tone), and neutralization did not produce a clear improvement compared to using the original text + tone label. Across both models, adding the tone label (“+tone”) sometimes helped slightly, but the dominant effect was that neutral rewriting removed useful cues and generally harmed factual classification accuracy.

Overall, these results suggest that (1) tone metadata is a mild auxiliary signal, but (2) rewriting into a neutral tone is risky because it may delete information the model uses to decide True/False on LIAR-style claims.

# External Domain Effects
Investigates whether augmenting statements with external domain knowledge or checkability signals improves model predictions. Tests the hypothesis that additional contextual information helps with fact verification.

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from transformers import pipeline

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# -----------------------------
# CONFIG
# -----------------------------
DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
SEP = "\t"
N_ROWS = None

STATEMENT_COL = "statement"

OUT_PATH = "1_Basics_Masterthesis/Results/preprocessed_test_with_ext_domain.csv"

# zero-shot model
ZS_MODEL = "facebook/bart-large-mnli"

# candidate domain labels
DOMAIN_LABELS = [
    "health", "economy", "politics", "education", "science",
    "technology", "environment", "sports", "law", "crime",
    "military", "immigration", "transportation", "energy", "culture",
    "foreign policy", "social issues", "business", "agriculture",
    "housing", "taxes", "welfare", "religion"
]

# Performance knobs
BATCH_SIZE = 16
MAX_LENGTH = 256 

# device: -1 CPU, 0 GPU0, 1 GPU1, ...
DEVICE = 0  # set -1 if no GPU

# -----------------------------
# MAIN
# -----------------------------
def run():
    df = pd.read_csv(DATA_PATH, sep=SEP)
    if N_ROWS is not None:
        df = df.iloc[:N_ROWS].copy()

    if STATEMENT_COL not in df.columns:
        raise ValueError(f"Missing column '{STATEMENT_COL}'. Found: {list(df.columns)}")

    df[STATEMENT_COL] = df[STATEMENT_COL].astype(str)

    classifier = pipeline(
        task="zero-shot-classification",
        model=ZS_MODEL,
        device=DEVICE
    )

    texts = df[STATEMENT_COL].tolist()

    domains = []
    scores = []

    # batching
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="External domain classification"):
        batch = texts[i:i + BATCH_SIZE]

        # pipeline supports list inputs
        results = classifier(
            batch,
            candidate_labels=DOMAIN_LABELS,
            multi_label=False,
            truncation=True,
            max_length=MAX_LENGTH
        )

        # if batch-size==1, pipeline may return dict; normalize to list
        if isinstance(results, dict):
            results = [results]

        for r in results:
            top_label = r["labels"][0]
            top_score = float(r["scores"][0])
            domains.append(top_label)
            scores.append(top_score)

    df["domain_ext"] = domains
    df["domain_ext_score"] = scores

    df.to_csv(OUT_PATH, index=False)
    print(f"\nSaved: {OUT_PATH}")
    print(df[["domain_ext", "domain_ext_score"]].head())

if __name__ == "__main__":
    run()


In [ ]:
import os
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
BASE_DATA_PATH = "1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
DOMAINS_PATH = "1_Basics_Masterthesis/Results/preprocessed_test_with_ext_domain.csv"
SEP = "\t"
N_ROWS = None

LABEL_COL = "label"
STATEMENT_COL = "statement"
HUMAN_SUBJECTS_COL = "subjects"

OUT_PRED_PATH = "1_Basics_Masterthesis/Results/ext_domain_effect_predictions.csv"
OUT_SUMMARY_PATH = "1_Basics_Masterthesis/Results/ext_domain_effect_summary.csv"

MODELS = {
    "google/gemma-7b-it": "causal_chat_or_instruct",
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
}

SUBJECT_MODES = [
    ("no_subjects", None),               
    ("human_subjects", "human_subjects"), 
    ("ext_domain_top1", "ext_domain_top1") 
]

# -----------------------------
# HELPERS
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

def get_single_token_id(tokenizer, variants):
    for v in variants:
        ids = tokenizer.encode(v, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return None

def true_false_token_ids(tokenizer):
    true_id = get_single_token_id(tokenizer, [" True", " true", "True", "true"])
    false_id = get_single_token_id(tokenizer, [" False", " false", "False", "false"])
    if true_id is None or false_id is None:
        raise ValueError("Could not find single-token IDs for True/False with this tokenizer.")
    return true_id, false_id

def next_token_probs(model, tokenizer, rendered_prompt: str, true_id: int, false_id: int):
    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits
    next_logits = logits[0, -1, :]
    probs = F.softmax(next_logits, dim=-1)
    return probs[true_id].item(), probs[false_id].item()

def decide_from_probs(p_true: float, p_false: float):
    pred_bin = 1 if p_true >= p_false else 0
    pred_str = "True" if pred_bin == 1 else "False"
    conf = max(p_true, p_false)
    return pred_bin, pred_str, conf

# -----------------------------
# PROMPTS
# -----------------------------
def P0_basic(statement: str, subjects: str | None) -> str:
    subj_block = f"Subjects: {subjects}\n" if subjects is not None and str(subjects).strip() != "" else ""
    return (
        subj_block +
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )

def P4_subjects_as_context_clues(statement: str, subjects: str | None) -> str:
    if subjects is not None and str(subjects).strip() != "":
        subj_sentence = f"You may use the following subjects as context clues: {subjects}.\n"
    else:
        subj_sentence = ""
    return (
        "You are an AI assistant specialized in fact-checking factual statements.\n"
        "Your task is to determine whether the following statement is factually correct.\n"
        + subj_sentence +
        "Respond strictly with one word: True or False.\n\n"
        "Example:\n"
        "Statement: The sky is green.\n"
        "Answer: False\n\n"
        f"Statement: {statement}\n"
        "Answer:"
    )

PROMPTS = {
    "P0_basic": P0_basic,
    "P4_subjects_as_context_clues": P4_subjects_as_context_clues,
}

# -----------------------------
# DATA LOAD
# -----------------------------
def load_merged():
    base = pd.read_csv(BASE_DATA_PATH, sep=SEP)
    dom = pd.read_csv(DOMAINS_PATH)

    if N_ROWS is not None:
        base = base.iloc[:N_ROWS].copy()
        dom = dom.iloc[:N_ROWS].copy()

    base = base.reset_index(drop=True)
    dom = dom.reset_index(drop=True)

    if STATEMENT_COL not in base.columns or LABEL_COL not in base.columns:
        raise ValueError(f"Base missing required columns. Found: {list(base.columns)}")

    if HUMAN_SUBJECTS_COL not in base.columns:
        base[HUMAN_SUBJECTS_COL] = ""

    if "domain_ext" not in dom.columns:
        raise ValueError(f"Domains file missing 'domain_top1'. Found: {list(dom.columns)}")

    out = base.copy()
    out["human_subjects"] = out[HUMAN_SUBJECTS_COL].astype(str)
    out["ext_domain_top1"] = dom["domain_ext"].astype(str)
    out[STATEMENT_COL] = out[STATEMENT_COL].astype(str)
    out[LABEL_COL] = out[LABEL_COL].astype(int)
    return out

# -----------------------------
# MAIN
# -----------------------------
def run():
    df = load_merged()

    rows = []
    summary_rows = []

    for model_name, model_type in MODELS.items():
        print(f"\n=== Loading: {model_name} ({model_type}) ===")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if device.type == "cuda" else None,
        ).to(device)
        model.eval()

        true_id, false_id = true_false_token_ids(tokenizer)

        for prompt_name, prompt_fn in PROMPTS.items():
            for subj_mode_name, subj_key in SUBJECT_MODES:
                desc = f"{model_name.split('/')[-1]} | {prompt_name} | {subj_mode_name}"
                iterator = tqdm(df.itertuples(index=True), total=len(df), desc=desc, leave=False)

                trues, preds, confs = [], [], []
                conf_correct, conf_wrong = [], []

                for r in iterator:
                    idx = r.Index
                    statement = getattr(r, STATEMENT_COL)
                    y_true = getattr(r, LABEL_COL)

                    subjects = None if subj_key is None else getattr(r, subj_key)

                    prompt = prompt_fn(statement, subjects)
                    rendered = maybe_apply_chat_template(tokenizer, prompt, model_type)

                    p_true, p_false = next_token_probs(model, tokenizer, rendered, true_id, false_id)
                    y_pred, y_pred_str, conf = decide_from_probs(p_true, p_false)

                    rows.append({
                        "idx": idx,
                        "model": model_name,
                        "model_short": model_name.split("/")[-1],
                        "prompt": prompt_name,
                        "subjects_mode": subj_mode_name,
                        "subjects_value": subjects if subjects is not None else "",
                        "y_true": y_true,
                        "y_pred": y_pred,
                        "y_pred_str": y_pred_str,
                        "confidence": conf,
                        "p_true": p_true,
                        "p_false": p_false,
                    })

                    trues.append(y_true)
                    preds.append(y_pred)
                    confs.append(conf)
                    (conf_correct if y_pred == y_true else conf_wrong).append(conf)

                acc = accuracy_score(trues, preds)
                macro_f1 = f1_score(trues, preds, average="macro")
                tn, fp, fn, tp = confusion_matrix(trues, preds, labels=[0, 1]).ravel()

                summary_rows.append({
                    "model": model_name,
                    "model_short": model_name.split("/")[-1],
                    "prompt": prompt_name,
                    "subjects_mode": subj_mode_name,
                    "accuracy": acc,
                    "macro_f1": macro_f1,
                    "n": len(trues),
                    "tn": tn, "fp": fp, "fn": fn, "tp": tp,
                    "mean_confidence": float(sum(confs) / len(confs)) if confs else None,
                    "mean_conf_correct": float(sum(conf_correct) / len(conf_correct)) if conf_correct else None,
                    "mean_conf_wrong": float(sum(conf_wrong) / len(conf_wrong)) if conf_wrong else None,
                })

        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(OUT_PRED_PATH, index=False)

    summary_df = pd.DataFrame(summary_rows).sort_values(["accuracy", "macro_f1"], ascending=False)
    summary_df.to_csv(OUT_SUMMARY_PATH, index=False)

    print(f"\nSaved predictions: {OUT_PRED_PATH}")
    print(f"Saved summary: {OUT_SUMMARY_PATH}")
    print("\nTop 20 summary rows:")
    print(summary_df.head(20).to_string(index=False))

if __name__ == "__main__":
    run()



=== Loading: google/gemma-7b-it (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

gemma-7b-it | P0_basic | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


gemma-7b-it | P0_basic | human_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

...

Llama-3.1-8B-Instruct | P0_basic | ext_domain_top1:   0%|          | 0/1267 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | human_subjects:   0%|          | 0/1267 [00:00<?, ?it/s…

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | ext_domain_top1:   0%|          | 0/1267 [00:00<?, ?it/…


Saved predictions: 1_Basics_Masterthesis/Results/ext_domain_effect_predictions.csv
Saved summary: 1_Basics_Masterthesis/Results/ext_domain_effect_summary.csv

Top 20 summary rows:
                           model           model_short                       prompt   subjects_mode  accuracy  macro_f1    n  tn  fp  fn  tp  mean_confidence  mean_conf_correct  mean_conf_wrong
              google/gemma-7b-it           gemma-7b-it                     P0_basic  human_subjects  0.611681  0.590369 1267 243 310 182 532         0.000017           0.000017         0.000017
              google/gemma-7b-it           gemma-7b-it P4_subjects_as_context_clues  human_subjects  0.610103  0.590531 1267 248 305 189 525         0.000021           0.000020         0.000022
              google/gemma-7b-it           gemma-7b-it                     P0_basic ext_domain_top1  0.607735  0.585731 1267 239 314 183 531         0.000016           0.000017         0.000016
              google/gemma-7b-it           

In [ ]:
import os
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# ENV
# -----------------------------
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)

# -----------------------------
# CONFIG
# -----------------------------
BASE_DATA_PATH = "LIAR/preprocessed_test_cleaned_binary.csv"
DOMAINS_PATH = "Results/preprocessed_test_with_ext_domain.csv"
SEP = "\t"
N_ROWS = None

LABEL_COL = "label"
STATEMENT_COL = "statement"
HUMAN_SUBJECTS_COL = "subjects"

OUT_PRED_PATH = "Results/ext_domain_effect_predictions_2.csv"
OUT_SUMMARY_PATH = "Results/ext_domain_effect_summary_2.csv"

MODELS = {
    "google/gemma-7b-it": "causal_chat_or_instruct",
    "meta-llama/Llama-3.1-8B-Instruct": "causal_chat_or_instruct",
}

SUBJECT_MODES = [
    ("no_subjects", None),               
    ("human_subjects", "human_subjects"), 
    ("ext_domain_top1", "ext_domain_top1") 
]

# -----------------------------
# HELPERS
# -----------------------------
def maybe_apply_chat_template(tokenizer, prompt: str, model_type: str) -> str:
    if model_type == "causal_chat_or_instruct" and hasattr(tokenizer, "apply_chat_template"):
        messages = [{"role": "user", "content": prompt}]
        try:
            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        except Exception:
            return prompt
    return prompt

def get_single_token_id(tokenizer, variants):
    for v in variants:
        ids = tokenizer.encode(v, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
    return None

def true_false_token_ids(tokenizer):
    true_id = get_single_token_id(tokenizer, [" True", " true", "True", "true"])
    false_id = get_single_token_id(tokenizer, [" False", " false", "False", "false"])
    if true_id is None or false_id is None:
        raise ValueError("Could not find single-token IDs for True/False with this tokenizer.")
    return true_id, false_id

def next_token_probs(model, tokenizer, rendered_prompt, true_id, false_id):
    inputs = tokenizer(rendered_prompt, return_tensors="pt", truncation=True).to(device)
    logits = model(**inputs).logits
    next_logits = logits[0, -1, :]
    
    tf_logits = torch.stack([next_logits[true_id], next_logits[false_id]])
    tf_probs = F.softmax(tf_logits, dim=-1)
    
    return tf_probs[0].item(), tf_probs[1].item()

def decide_from_probs(p_true: float, p_false: float):
    pred_bin = 1 if p_true >= p_false else 0
    pred_str = "True" if pred_bin == 1 else "False"
    conf = max(p_true, p_false)
    return pred_bin, pred_str, conf

# -----------------------------
# PROMPTS
# -----------------------------
def P0_basic(statement: str, subjects: str | None) -> str:
    subj_block = f"Subjects: {subjects}\n" if subjects is not None and str(subjects).strip() != "" else ""
    return (
        subj_block +
        f"Statement: {statement}\n"
        f"Is this statement true or false?\n"
        f"Answer with only one word: True or False.\n"
        f"Answer:"
    )

def P4_subjects_as_context_clues(statement: str, subjects: str | None) -> str:
    if subjects is not None and str(subjects).strip() != "":
        subj_sentence = f"You may use the following subjects as context clues: {subjects}.\n"
    else:
        subj_sentence = ""
    return (
        "You are an AI assistant specialized in fact-checking factual statements.\n"
        "Your task is to determine whether the following statement is factually correct.\n"
        + subj_sentence +
        "Respond strictly with one word: True or False.\n\n"
        "Example:\n"
        "Statement: The sky is green.\n"
        "Answer: False\n\n"
        f"Statement: {statement}\n"
        "Answer:"
    )

PROMPTS = {
    "P0_basic": P0_basic,
    "P4_subjects_as_context_clues": P4_subjects_as_context_clues,
}

# -----------------------------
# DATA LOAD
# -----------------------------
def load_merged():
    base = pd.read_csv(BASE_DATA_PATH, sep=SEP)
    dom = pd.read_csv(DOMAINS_PATH)

    if N_ROWS is not None:
        base = base.iloc[:N_ROWS].copy()
        dom = dom.iloc[:N_ROWS].copy()

    base = base.reset_index(drop=True)
    dom = dom.reset_index(drop=True)

    if STATEMENT_COL not in base.columns or LABEL_COL not in base.columns:
        raise ValueError(f"Base missing required columns. Found: {list(base.columns)}")

    if HUMAN_SUBJECTS_COL not in base.columns:
        base[HUMAN_SUBJECTS_COL] = ""

    if "domain_ext" not in dom.columns:
        raise ValueError(f"Domains file missing 'domain_top1'. Found: {list(dom.columns)}")

    out = base.copy()
    out["human_subjects"] = out[HUMAN_SUBJECTS_COL].astype(str)
    out["ext_domain_top1"] = dom["domain_ext"].astype(str)
    out[STATEMENT_COL] = out[STATEMENT_COL].astype(str)
    out[LABEL_COL] = out[LABEL_COL].astype(int)
    return out

# -----------------------------
# MAIN
# -----------------------------
def run():
    df = load_merged()

    rows = []
    summary_rows = []

    for model_name, model_type in MODELS.items():
        print(f"\n=== Loading: {model_name} ({model_type}) ===")
        tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            dtype=torch.float16 if device.type == "cuda" else None,
        ).to(device)
        model.eval()

        true_id, false_id = true_false_token_ids(tokenizer)

        for prompt_name, prompt_fn in PROMPTS.items():
            for subj_mode_name, subj_key in SUBJECT_MODES:
                desc = f"{model_name.split('/')[-1]} | {prompt_name} | {subj_mode_name}"
                iterator = tqdm(df.itertuples(index=True), total=len(df), desc=desc, leave=False)

                trues, preds, confs = [], [], []
                conf_correct, conf_wrong = [], []

                for r in iterator:
                    idx = r.Index
                    statement = getattr(r, STATEMENT_COL)
                    y_true = getattr(r, LABEL_COL)

                    subjects = None if subj_key is None else getattr(r, subj_key)

                    prompt = prompt_fn(statement, subjects)
                    rendered = maybe_apply_chat_template(tokenizer, prompt, model_type)

                    p_true, p_false = next_token_probs(model, tokenizer, rendered, true_id, false_id)
                    y_pred, y_pred_str, conf = decide_from_probs(p_true, p_false)

                    rows.append({
                        "idx": idx,
                        "model": model_name,
                        "model_short": model_name.split("/")[-1],
                        "prompt": prompt_name,
                        "subjects_mode": subj_mode_name,
                        "subjects_value": subjects if subjects is not None else "",
                        "y_true": y_true,
                        "y_pred": y_pred,
                        "y_pred_str": y_pred_str,
                        "confidence": conf,
                        "p_true": p_true,
                        "p_false": p_false,
                    })

                    trues.append(y_true)
                    preds.append(y_pred)
                    confs.append(conf)
                    (conf_correct if y_pred == y_true else conf_wrong).append(conf)

                acc = accuracy_score(trues, preds)
                macro_f1 = f1_score(trues, preds, average="macro")
                tn, fp, fn, tp = confusion_matrix(trues, preds, labels=[0, 1]).ravel()

                summary_rows.append({
                    "model": model_name,
                    "model_short": model_name.split("/")[-1],
                    "prompt": prompt_name,
                    "subjects_mode": subj_mode_name,
                    "accuracy": acc,
                    "macro_f1": macro_f1,
                    "n": len(trues),
                    "tn": tn, "fp": fp, "fn": fn, "tp": tp,
                    "mean_confidence": float(sum(confs) / len(confs)) if confs else None,
                    "mean_conf_correct": float(sum(conf_correct) / len(conf_correct)) if conf_correct else None,
                    "mean_conf_wrong": float(sum(conf_wrong) / len(conf_wrong)) if conf_wrong else None,
                })

        del model, tokenizer
        if device.type == "cuda":
            torch.cuda.empty_cache()

    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(OUT_PRED_PATH, index=False)

    summary_df = pd.DataFrame(summary_rows).sort_values(["accuracy", "macro_f1"], ascending=False)
    summary_df.to_csv(OUT_SUMMARY_PATH, index=False)

    print(f"\nSaved predictions: {OUT_PRED_PATH}")
    print(f"Saved summary: {OUT_SUMMARY_PATH}")
    print("\nTop 20 summary rows:")
    print(summary_df.head(20).to_string(index=False))

if __name__ == "__main__":
    run()



=== Loading: google/gemma-7b-it (causal_chat_or_instruct) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

gemma-7b-it | P0_basic | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


gemma-7b-it | P0_basic | human_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

...

Llama-3.1-8B-Instruct | P0_basic | ext_domain_top1:   0%|          | 0/1267 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | no_subjects:   0%|          | 0/1267 [00:00<?, ?it/s]

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | human_subjects:   0%|          | 0/1267 [00:00<?, ?it/s…

Llama-3.1-8B-Instruct | P4_subjects_as_context_clues | ext_domain_top1:   0%|          | 0/1267 [00:00<?, ?it/…


Saved predictions: Results/ext_domain_effect_predictions_2.csv
Saved summary: Results/ext_domain_effect_summary_2.csv

Top 20 summary rows:
                           model           model_short                       prompt   subjects_mode  accuracy  macro_f1    n  tn  fp  fn  tp  mean_confidence  mean_conf_correct  mean_conf_wrong
              google/gemma-7b-it           gemma-7b-it                     P0_basic  human_subjects  0.611681  0.590369 1267 243 310 182 532         0.955489           0.963280         0.943217
              google/gemma-7b-it           gemma-7b-it P4_subjects_as_context_clues  human_subjects  0.610103  0.590531 1267 248 305 189 525         0.965080           0.972270         0.953829
              google/gemma-7b-it           gemma-7b-it                     P0_basic ext_domain_top1  0.607735  0.585731 1267 239 314 183 531         0.961258           0.970235         0.947350
              google/gemma-7b-it           gemma-7b-it                     P0_basic

## Conclusion

External domain classification brought no consistent improvement over using the original (human) LIAR subjects. For Gemma-7B-IT, the best performance still comes from human_subjects (P0: 0.6117, P4: 0.6101). Replacing them with the external top-1 domain slightly reduced accuracy (P0: 0.6077, P4: 0.6022) and is roughly comparable to no_subjects (P0: 0.6062, P4: 0.6014).
For Llama-3.1-8B-Instruct, adding human_subjects or external domains also did not help; the strongest setting is actually P4 with no_subjects (0.4917), while human/external subjects remain lower (≈ 0.466–0.481). Overall, external domains are weaker than human subjects and do not reliably outperform the no-subject baseline, so they’re mainly useful if you need an automatic subject signal when human labels are unavailable.